# 00. Fuentes y adquisición de datos electorales



Este cuaderno reúne y verifica las fuentes electorales y demográficas de Barrancabermeja antes de su análisis.

## Alcance

Incluye elecciones de **Alcaldía y Concejo** (2023, 2019 y 2015), **Presidencia** (2026, 2022 y 2018), **Congreso** (2026 y 2022), censos electorales, DIVIPOL, candidaturas, población municipal del DANE y cartografía. El registro distingue entre fuentes identificadas, archivos originales verificados y tablas locales interpretadas. La descarga de un archivo no significa que los datos estén listos para análisis.

El piloto de Presidencia 2026, primera vuelta, se interpreta con el PDF de estructuras incluido en el ZIP y se valida con las huellas digitales, DIVIPOL y novedades. Los demás archivos se procesan cuando su estructura y cobertura territorial pueden verificarse. Los formatos nuevos o ambiguos quedan pendientes hasta contar con evidencia suficiente.

El cuaderno 01 se ejecuta cuando las fuentes centrales y sus tablas locales están verificadas, y las fuentes ampliadas tienen un estado documentado.

## 1. Configuración y directorios



In [1]:
from __future__ import annotations

import csv
import hashlib
import html
import io
import json
import os
import re
import shutil
import tempfile
import time
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from datetime import datetime, timezone
from html.parser import HTMLParser
from pathlib import Path, PurePosixPath
from xml.etree import ElementTree

import pandas as pd

def raiz_proyecto(inicio: Path) -> Path:
    for carpeta in (inicio.resolve(), *inicio.resolve().parents):
        if (carpeta / "datos").is_dir() and (carpeta / "notebooks").is_dir():
            return carpeta
    raise FileNotFoundError("Abre el cuaderno desde la raíz o notebooks/ del proyecto (carpetas datos/ y notebooks/).")

RAIZ = raiz_proyecto(Path.cwd())
BRUTOS = RAIZ / "datos" / "brutos"
SALIDAS = RAIZ / "docs" / "fuentes" / "auditorias" / "00_fuentes_y_adquisicion"
SALIDAS.mkdir(parents=True, exist_ok=True)
SIN_RED = os.getenv("OBSERVATORIO_SIN_RED", "0") == "1"
DESCARGAR = not SIN_RED
MAX_GIB_POR_ARCHIVO = 3
MAX_GIB_DESCOMPRIMIDOS = 12
RESERVA_DISCO_GIB = 1
VALIDAR_CRC_COMPLETO = True
USER_AGENT = "ObservatorioElectoralBarrancabermeja/1.0 (investigacion; fuentes publicas)"
FECHA_INICIO = datetime.now(timezone.utc)
ID_EJECUCION = FECHA_INICIO.strftime("%Y%m%d_%H%M%S_%f")
AUDITORIA = SALIDAS / ID_EJECUCION
AUDITORIA.mkdir(exist_ok=False)
print("Proyecto:", RAIZ)
print("Descarga en red:", DESCARGAR, "| Auditoría:", AUDITORIA.relative_to(RAIZ))

Proyecto: D:\Users\LAURA PEREZ\Desktop\CIENCIA DE DATOS\Proy_Electoral_Bca
Descarga en red: True | Auditoría: docs\fuentes\auditorias\00_fuentes_y_adquisicion\20261003_164243_810157


## 2. Catálogo y lista de trabajo

La matriz incluye fuentes centrales y complementarias. Las URL confirmadas provienen del [histórico oficial de la Registraduría](https://observatorio.registraduria.gov.co/views/electoral/historicos-resultados.php) y de las [series municipales del DANE](https://www.dane.gov.co/index.php/estadisticas-por-tema/demografia-y-poblacion/proyecciones-de-poblacion). El portal no muestra enlaces directos para las elecciones territoriales de 2019 y 2023. El [censo electoral histórico](https://www.registraduria.gov.co/-Consulta-los-censos-historicos-utilizados-para-los-procesos-electorales-desde-.html) se consulta de forma interactiva. El catálogo no genera URL cuando el enlace no está confirmado.

`obligatoria=True` identifica las fuentes del corpus central; no confirma que el archivo esté disponible. Las fuentes complementarias también se registran. Antes de agregar un archivo manual, comprueba que corresponda al año y a la elección indicados.

In [2]:
HIST = "https://observatorio.registraduria.gov.co/views/electoral/historicos-resultados.php"
DANE = "https://www.dane.gov.co/index.php/estadisticas-por-tema/demografia-y-poblacion/proyecciones-de-poblacion"
CENSO = "https://www.registraduria.gov.co/-Consulta-los-censos-historicos-utilizados-para-los-procesos-electorales-desde-.html"
BASE = "https://observatorio.registraduria.gov.co/anexos/"

def fuente(id: str, titulo: str, familia: str, anio: int | None, pagina: str,
           url: str = "", extension: str = "zip", obligatoria: bool = True,
           espera_tabla_local: bool = True, nota: str = "") -> dict:
    return dict(id_fuente=id, titulo=titulo, familia=familia, anio=anio,
                pagina_oficial=pagina, url_directa=url, extension=extension,
                obligatoria=obligatoria, espera_tabla_local=espera_tabla_local, nota=nota)

FUENTES = [
    fuente("REG_2026_PRES_1V", "Presidencia 2026 · 1V", "presidencia", 2026, HIST,
           BASE + "MMV_Presidente1V_2026.zip", nota="Piloto interpretado y conciliado en 00"),
    fuente("REG_2026_PRES_2V", "Presidencia 2026 · 2V", "presidencia", 2026, HIST,
           BASE + "MMV_Presidente2V_2026.zip"),
    fuente("REG_2026_CONGRESO", "Congreso 2026 · Senado/Cámara", "congreso", 2026, HIST,
           BASE + "MMV_CONGRESO_2026.zip", nota="El visor describe CSV por departamento dentro del recurso"),
    fuente("REG_2023_TERR", "Territoriales 2023 · Santander · Alcaldía/Concejo", "territorial", 2023, HIST,
           extension="csv", nota="Selector departamental: URL exacta de Santander por resolver; admite CSV o ZIP verificado"),
    fuente("REG_2019_TERR", "Territoriales 2019 · Santander · Alcaldía/Concejo", "territorial", 2019, HIST,
           extension="csv", nota="Selector departamental: URL exacta de Santander por resolver; admite CSV o ZIP verificado"),
    fuente("REG_2022_PRES_1V", "Presidencia 2022 · 1V", "presidencia", 2022, HIST,
           BASE + "MMV_NACIONAL_PRESIDENTE_2022_1v.zip"),
    fuente("REG_2022_PRES_2V", "Presidencia 2022 · 2V", "presidencia", 2022, HIST,
           BASE + "MMV_NACIONAL_PRESIDENTE_2022_2v.zip"),
    fuente("REG_2022_CONGRESO", "Congreso 2022 · Santander", "congreso", 2022, HIST,
           "https://observatorio.registraduria.gov.co/views/electoral/MMV_CONGRESO_2022_SANTANDER.zip",
           extension="zip", nota="Enlace de Santander visible en el portal, pero su respuesta fue HTML en la ejecución; revisar selector/descarga manual"),
    fuente("REG_CENSO_HIST", "Censo electoral histórico por puesto", "censo_electoral", None, CENSO,
           extension="csv", nota="Consulta interactiva; exportar años y elecciones de interés"),
    fuente("DANE_MUN_EDAD_2018_2042", "Población municipal por área, sexo y edad · 2018-2042", "poblacion", None, DANE,
           "https://www.dane.gov.co/files/censo2018/proyecciones-de-poblacion/Municipal/PPED-AreaSexoEdadMun-2018-2042_VP.xlsx",
           extension="xlsx"),
    fuente("DANE_MUN_EDAD_2005_2017", "Población municipal por área, sexo y edad · 2005-2017", "poblacion", None, DANE,
           "https://www.dane.gov.co/files/censo2018/proyecciones-de-poblacion/Municipal/DCD-area-sexo-edad-proypoblacion-Mun-2005-2017_VP.xlsx",
           extension="xlsx"),
    fuente("REG_2015_TERR", "Territoriales 2015 · legado", "territorial", 2015, HIST,
           obligatoria=False, nota="Visor heredado; exportación tabular por verificar"),
    fuente("REG_2018_CONGRESO", "Congreso 2018 · Santander · legado", "congreso", 2018, HIST,
           extension="csv", obligatoria=False, nota="Consultar visor histórico y exportación departamental"),
    fuente("REG_2018_PRES_1V", "Presidencia 2018 · 1V", "presidencia", 2018, HIST,
           BASE + "MMV_NACIONAL_PRESIDENTE_2018_1v.zip", obligatoria=False),
    fuente("REG_2018_PRES_2V", "Presidencia 2018 · 2V", "presidencia", 2018, HIST,
           BASE + "MMV_NACIONAL_PRESIDENTE_2018_2v.zip", obligatoria=False),
    fuente("DANE_CNPV_2018", "CNPV 2018 · características geográficas y sociodemográficas", "censo_poblacion", 2018,
           "https://geoportal.dane.gov.co/geovisores/sociedad/cnpv2018-detallado/", "", "csv", False,
           nota="Resolver descarga desagregada y metadatos de cobertura espacial"),
    fuente("REG_CANDIDATURAS_2019", "Candidaturas inscritas · Alcaldía/Concejo 2019", "candidaturas", 2019,
           "https://wapp.registraduria.gov.co/electoral/Elecciones-2019/infocandidatos2019.php",
           extension="csv", nota="Exportar candidaturas definitivas de 2019; incluir personas con cero votos"),
    fuente("REG_CANDIDATURAS_2023", "Candidaturas inscritas · Alcaldía/Concejo 2023", "candidaturas", 2023,
           "https://wapp.registraduria.gov.co/electoral/elecciones-territoriales-2023/",
           extension="csv", nota="Exportar candidaturas definitivas de 2023; incluir personas con cero votos"),
    fuente("GEO_COMUNAS", "Límites oficiales de comunas y corregimientos", "cartografia", None,
           "https://www.barrancabermeja.gov.co/publicaciones/21/mapas/", "", "zip", False,
           nota="Por resolver capa, año, escala y sistema de referencia"),
]

# Solo añadir una URL aquí cuando se haya copiado del enlace de descarga oficial.
# Ejemplo: "REG_2023_TERR": "https://observatorio.registraduria.gov.co/.../archivo_santander.csv"
ENLACES_MANUALES: dict[str, str] = {}

# Si una fuente usa un selector o el proveedor bloquea la descarga automatizada,
# coloca su archivo exportado BAJO datos/brutos/ y registra la ruta relativa.
# Ejemplo: "REG_2023_TERR": "datos/brutos/territorial/2023/santander_2023.csv"
ARCHIVOS_MANUALES: dict[str, str] = {}

# Para un selector sin URL directa: registrar la página y fecha de la descarga.
# EVIDENCIA_MANUAL["REG_2023_TERR"] = {
#     "pagina_oficial": HIST, "fecha_descarga_utc": "2026-09-29T15:00:00Z"}
# Si el navegador muestra URL exacta: usa "url_descarga" (dominio oficial).
EVIDENCIA_MANUAL: dict[str, dict[str, str]] = {}

ids = [f["id_fuente"] for f in FUENTES]
assert len(ids) == len(set(ids)), "El contrato tiene IDs duplicados"
catalogo = pd.DataFrame(FUENTES)
print("Fuentes:", len(catalogo), "| Obligatorias:", int(catalogo.obligatoria.sum()),
      "| URLs catalogadas:", int(catalogo.url_directa.ne("").sum()))
display(catalogo[["id_fuente", "familia", "anio", "obligatoria", "url_directa", "nota"]])

Fuentes: 19 | Obligatorias: 13 | URLs catalogadas: 10


,id_fuente,familia,anio,obligatoria,url_directa,nota
0,REG_2026_PRES_1V,presidencia,2026.0,True,https://observatorio.registraduria.gov.co/anex...,Piloto interpretado y conciliado en 00
1,REG_2026_PRES_2V,presidencia,2026.0,True,https://observatorio.registraduria.gov.co/anex...,
2,REG_2026_CONGRESO,congreso,2026.0,True,https://observatorio.registraduria.gov.co/anex...,El visor describe CSV por departamento dentro ...
3,REG_2023_TERR,territorial,2023.0,True,,Selector departamental: URL exacta de Santande...
4,REG_2019_TERR,territorial,2019.0,True,,Selector departamental: URL exacta de Santande...
5,REG_2022_PRES_1V,presidencia,2022.0,True,https://observatorio.registraduria.gov.co/anex...,
6,REG_2022_PRES_2V,presidencia,2022.0,True,https://observatorio.registraduria.gov.co/anex...,
7,REG_2022_CONGRESO,congreso,2022.0,True,https://observatorio.registraduria.gov.co/view...,"Enlace de Santander visible en el portal, pero..."
8,REG_CENSO_HIST,censo_electoral,NaN,True,,Consulta interactiva; exportar años y eleccion...
9,DANE_MUN_EDAD_2018_2042,poblacion,NaN,True,https://www.dane.gov.co/files/censo2018/proyec...,


## 3. Resolver descargas mediante el portal oficial

El cuaderno busca enlaces en el portal y los asigna a Santander solo cuando el año y el departamento se identifican sin ambigüedad. Los enlaces dudosos se muestran como candidatos. También admite URL verificadas y archivos descargados desde el selector oficial. Si una página bloquea la consulta, queda registrada y el proceso continúa con las demás fuentes.

In [3]:
class ExtractorEnlaces(HTMLParser):
    def __init__(self):
        super().__init__()
        self.valores: list[str] = []

    def handle_starttag(self, tag, attrs):
        if tag not in {"a", "option", "button"}:
            return
        for clave, valor in attrs:
            if valor and clave in {"href", "value", "data-url", "data-href", "onclick"}:
                self.valores.append(valor)

def dominio_oficial(url: str, familia: str) -> bool:
    parsed = urllib.parse.urlparse(url)
    host = (parsed.hostname or "").lower()
    raiz = "dane.gov.co" if familia in {"poblacion", "censo_poblacion"} else "registraduria.gov.co"
    return parsed.scheme == "https" and (host == raiz or host.endswith("." + raiz))

def descubrir_links(url_pagina: str) -> tuple[list[str], str]:
    if SIN_RED:
        return [], "Exploración de página desactivada por OBSERVATORIO_SIN_RED=1"
    try:
        req = urllib.request.Request(url_pagina, headers={"User-Agent": USER_AGENT})
        with urllib.request.urlopen(req, timeout=45) as respuesta:
            if respuesta.status != 200:
                raise ValueError(f"HTTP {respuesta.status}")
            raw = respuesta.read(4 * 1024 * 1024 + 1)
            if len(raw) > 4 * 1024 * 1024:
                raise ValueError("La página excede el límite de inspección de 4 MiB")
        pagina = raw.decode("utf-8", errors="replace")
        parser = ExtractorEnlaces()
        parser.feed(pagina)
        posibles = parser.valores + re.findall(r'https?[^\s"\'<>]+?\.(?:csv|zip|xlsx)(?:\?[^\s"\'<>]*)?', pagina, re.I)
        enlaces = set()
        for valor in posibles:
            valor = html.unescape(valor).replace(r"\/", "/")
            # Evita interpretar JavaScript arbitrario como una URL descargable.
            if re.search(r"[\s'\"()]", valor):
                continue
            absoluto = urllib.parse.urljoin(url_pagina, valor)
            if dominio_oficial(absoluto, "electoral") and re.search(r"\.(csv|zip|xlsx)(?:$|\?)", absoluto, re.I):
                enlaces.add(absoluto)
        return sorted(enlaces), "accesible"
    except (OSError, ValueError, urllib.error.URLError, urllib.error.HTTPError) as exc:
        return [], f"{type(exc).__name__}: {exc}"

enlaces_visibles, estado_portal = descubrir_links(HIST)
enlaces_santander = [x for x in enlaces_visibles if "santander" in x.lower()]
descubrimientos = []
for id_fuente, anio in [("REG_2023_TERR", 2023), ("REG_2019_TERR", 2019), ("REG_2022_CONGRESO", 2022)]:
    c = [x for x in enlaces_santander if str(anio) in x and (
        id_fuente != "REG_2022_CONGRESO" or
        PurePosixPath(urllib.parse.urlparse(x).path).name.upper() == "MMV_CONGRESO_2022_SANTANDER.ZIP")]
    descubrimientos.append({"id_fuente": id_fuente, "candidatos": c,
                            "url_unica": c[0] if len(c) == 1 else "",
                            "estado": "unica" if len(c) == 1 else ("ambigua" if c else "no_resuelta")})
for f in FUENTES:
    if f["id_fuente"] in ENLACES_MANUALES:
        url = ENLACES_MANUALES[f["id_fuente"]]
        if not dominio_oficial(url, f["familia"]):
            raise ValueError(f"URL manual no pertenece al proveedor oficial: {f['id_fuente']}")
        f["url_directa"] = url
    elif not f["url_directa"]:
        match = next((d for d in descubrimientos if d["id_fuente"] == f["id_fuente"]), None)
        if match and match["url_unica"]:
            f["url_directa"] = match["url_unica"]

catalogo = pd.DataFrame(FUENTES)
(AUDITORIA / "links_visibles_portal.json").write_text(json.dumps({
    "url_pagina": HIST, "estado": estado_portal, "enlaces": enlaces_visibles,
    "resolucion_santander": descubrimientos}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Portal:", estado_portal, "| enlaces a archivos visibles:", len(enlaces_visibles))
display(pd.DataFrame(descubrimientos))
# Los selectores pueden entregar CSV, ZIP o XLSX. La firma se comprueba al adquirir.
for f in FUENTES:
    if f["id_fuente"] in {"REG_2023_TERR", "REG_2019_TERR", "REG_2022_CONGRESO",
                          "REG_2018_CONGRESO", "REG_2015_TERR", "REG_CENSO_HIST",
                          "REG_CANDIDATURAS_2019", "REG_CANDIDATURAS_2023",
                          "DANE_CNPV_2018", "GEO_COMUNAS"}:
        recurso = ARCHIVOS_MANUALES.get(f["id_fuente"], f["url_directa"])
        sufijo = PurePosixPath(urllib.parse.urlparse(recurso).path).suffix.lower().lstrip(".")
        if sufijo in {"csv", "zip", "xlsx"}:
            f["extension"] = sufijo
catalogo = pd.DataFrame(FUENTES)

Portal: accesible | enlaces a archivos visibles: 41


,id_fuente,candidatos,url_unica,estado
0,REG_2023_TERR,[],,no_resuelta
1,REG_2019_TERR,[],,no_resuelta
2,REG_2022_CONGRESO,[https://observatorio.registraduria.gov.co/vie...,https://observatorio.registraduria.gov.co/view...,unica


## 4. Descargar y conservar los originales

Los archivos se descargan uno a uno. El proceso comprueba el tamaño, el espacio disponible, la integridad de ZIP y XLSX, el SHA-256 y que la respuesta no sea una página HTML. La descarga puede repetirse sin reemplazar un original válido. Un ZIP íntegro puede contener varias tablas o tipos de resultado. El archivo de Presidencia 2026, primera vuelta, también se compara con la huella ya verificada.

In [4]:
HASH_ZIP_PILOTO = "e8623fdc10d0f0a37ffb984e477f26ce8315759596fdc7837729de86e9215642"

def sha256_archivo(ruta: Path) -> str:
    dig = hashlib.sha256()
    with ruta.open("rb") as stream:
        for bloque in iter(lambda: stream.read(1024 * 1024), b""):
            dig.update(bloque)
    return dig.hexdigest()

def revisar_formato(ruta: Path, extension: str) -> dict:
    if not ruta.is_file() or ruta.stat().st_size == 0:
        raise ValueError("archivo vacío o ausente")
    with ruta.open("rb") as s:
        cabecera = s.read(4096)
    if b"<html" in cabecera.lower() or b"<!doctype html" in cabecera.lower():
        raise ValueError("la respuesta es una página HTML, no datos")
    if extension in {"zip", "xlsx"}:
        if not zipfile.is_zipfile(ruta):
            raise ValueError("firma ZIP/XLSX ausente")
        with zipfile.ZipFile(ruta) as z:
            miembros = [x for x in z.infolist() if not x.is_dir()]
            if not miembros:
                raise ValueError("archivo ZIP vacío")
            if extension == "xlsx" and not {"[Content_Types].xml", "xl/workbook.xml"}.issubset(z.namelist()):
                raise ValueError("XLSX sin estructura OOXML mínima")
            total_abierto = sum(x.file_size for x in miembros)
            if total_abierto > MAX_GIB_DESCOMPRIMIDOS * 1024**3:
                raise ValueError("expansión ZIP superior al límite configurado")
            if VALIDAR_CRC_COMPLETO and z.testzip() is not None:
                raise ValueError("falló CRC de un miembro ZIP")
            return {"miembros": len(miembros), "suma_bytes_descomprimidos": sum(x.file_size for x in miembros)}
    if extension == "csv":
        if b"\x00" in cabecera[:2048] or cabecera.startswith(b"PK\x03\x04"):
            raise ValueError("CSV incompatible con la firma observada")
        return {"miembros": None, "suma_bytes_descomprimidos": None}
    raise ValueError(f"extensión sin validador: {extension}")

def solicitar(url: str):
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT}, method="GET")
    for intento in range(2):
        try:
            return urllib.request.urlopen(req, timeout=90)
        except urllib.error.HTTPError as exc:
            if exc.code not in (429, 500, 502, 503, 504) or intento == 1:
                raise
        except (OSError, urllib.error.URLError, TimeoutError):
            if intento == 1:
                raise
        time.sleep(2 + intento)

def adquirir(f: dict) -> dict:
    nombre = PurePosixPath(urllib.parse.urlparse(f["url_directa"]).path).name if f["url_directa"] else ""
    destino = BRUTOS / f["familia"] / (str(f["anio"]) if f["anio"] else "serie") / (f["id_fuente"] + "_" + nombre)
    info = {"id_fuente": f["id_fuente"], "url_origen": f["url_directa"],
            "fecha_utc": datetime.now(timezone.utc).isoformat(),
            "ruta_relativa": "", "estado": "", "http": None, "bytes": None,
            "sha256": "", "miembros": None, "suma_bytes_descomprimidos": None,
            "detalle": "", "estado_procedencia": "pendiente", "fecha_descarga_declarada": ""}
    if f["id_fuente"] in ARCHIVOS_MANUALES:
        manual = (RAIZ / ARCHIVOS_MANUALES[f["id_fuente"]]).resolve()
        try:
            manual.relative_to(BRUTOS.resolve())
            if manual.suffix.lower() != "." + f["extension"]:
                raise ValueError(f"la extensión debe ser .{f['extension']} según el contrato")
            extra = revisar_formato(manual, f["extension"])
            huella = sha256_archivo(manual)
            if f["id_fuente"] == "REG_2026_PRES_1V" and huella != HASH_ZIP_PILOTO:
                raise ValueError("el original 2026 1V difiere del SHA-256 auditado")
            evidencia = EVIDENCIA_MANUAL.get(f["id_fuente"], {})
            url_declarada = evidencia.get("url_descarga", f["url_directa"])
            fecha_declarada = evidencia.get("fecha_descarga_utc", "")
            pagina_declarada = evidencia.get("pagina_oficial", "")
            if url_declarada and not dominio_oficial(url_declarada, f["familia"]):
                raise ValueError("URL de procedencia manual fuera del dominio oficial")
            if fecha_declarada:
                datetime.fromisoformat(fecha_declarada.replace("Z", "+00:00"))
            if pagina_declarada and pagina_declarada != f["pagina_oficial"]:
                raise ValueError("La página declarada no coincide con el catálogo")
            info["estado_procedencia"] = ("url_oficial_declarada" if url_declarada else
                "pagina_fecha_declaradas" if pagina_declarada and fecha_declarada else "pendiente")
            info["fecha_descarga_declarada"] = fecha_declarada
            info["url_origen"] = url_declarada or pagina_declarada
            info.update(estado="manual_validado", ruta_relativa=manual.relative_to(RAIZ).as_posix(),
                        bytes=manual.stat().st_size, sha256=huella, **extra,
                        detalle="Archivo aportado manualmente; revisar URL/fecha de descarga en la documentación")
        except (OSError, ValueError, zipfile.BadZipFile) as exc:
            info.update(estado="manual_invalido", detalle=f"{type(exc).__name__}: {exc}")
        return info
    if f["id_fuente"] == "REG_2026_PRES_1V":
        # Ruta usada por el cuaderno 00: se reutiliza sin mover el original.
        existente = BRUTOS / "registraduria" / "2026" / "MMV_Presidente1V_2026.zip"
        if existente.exists():
            destino = existente
    if destino.exists():
        try:
            extra = revisar_formato(destino, f["extension"])
            huella = sha256_archivo(destino)
            if f["id_fuente"] == "REG_2026_PRES_1V" and huella != HASH_ZIP_PILOTO:
                raise ValueError("el original 2026 1V difiere del SHA-256 auditado")
            info["estado_procedencia"] = "url_oficial_catalogada"
            info.update(estado="existente_validado", ruta_relativa=destino.relative_to(RAIZ).as_posix(),
                        bytes=destino.stat().st_size, sha256=huella, **extra)
        except (OSError, ValueError, zipfile.BadZipFile) as exc:
            info.update(estado="existente_invalido", ruta_relativa=destino.relative_to(RAIZ).as_posix(),
                        detalle=f"{type(exc).__name__}: {exc}")
        return info
    if not f["url_directa"]:
        info.update(estado="sin_url_resuelta", detalle=f["nota"])
        return info
    if not dominio_oficial(f["url_directa"], f["familia"]):
        info.update(estado="url_no_oficial", detalle="El host no coincide con el organismo de la fuente")
        return info
    if not DESCARGAR:
        info.update(estado="pendiente_modo_sin_red", detalle="Descarga desactivada para esta ejecución")
        return info
    limite = MAX_GIB_POR_ARCHIVO * 1024**3
    reserva = RESERVA_DISCO_GIB * 1024**3
    temporal = None
    fase = "solicitud"
    try:
        destino.parent.mkdir(parents=True, exist_ok=True)
        if shutil.disk_usage(destino.parent).free < reserva + 128 * 1024**2:
            raise OSError("espacio libre inferior a la reserva configurada")
        with solicitar(f["url_directa"]) as respuesta:
            info["http"] = respuesta.status
            if respuesta.status != 200:
                raise ValueError(f"respuesta HTTP {respuesta.status}, se esperaba 200")
            esperado = respuesta.headers.get("Content-Length", "")
            esperado = int(esperado) if esperado.isdigit() else None
            if esperado is not None and esperado > limite:
                raise ValueError("archivo mayor que MAX_GIB_POR_ARCHIVO")
            if esperado is not None and shutil.disk_usage(destino.parent).free < esperado + reserva:
                raise OSError("espacio libre insuficiente para archivo y reserva")
            fase = "transferencia"
            with tempfile.NamedTemporaryFile(dir=destino.parent, suffix=".part", delete=False) as tmp:
                temporal = Path(tmp.name)
                total = 0
                for bloque in iter(lambda: respuesta.read(1024 * 1024), b""):
                    total += len(bloque)
                    if total > limite or shutil.disk_usage(destino.parent).free < reserva:
                        raise OSError("límite de tamaño o reserva de disco alcanzados")
                    tmp.write(bloque)
        fase = "integridad"
        if esperado is not None and total != esperado:
            raise ValueError(f"transferencia incompleta: {total} de {esperado} bytes")
        extra = revisar_formato(temporal, f["extension"])
        huella = sha256_archivo(temporal)
        if destino.exists():
            raise FileExistsError("otro proceso creó el destino; no se sobrescribió")
        temporal.replace(destino)
        temporal = None
        info["estado_procedencia"] = "url_oficial_descargada"
        info.update(estado="descargado_validado", ruta_relativa=destino.relative_to(RAIZ).as_posix(),
                    bytes=total, sha256=huella, **extra)
    except (OSError, ValueError, urllib.error.HTTPError, urllib.error.URLError,
            zipfile.BadZipFile, TimeoutError) as exc:
        info.update(estado="error_adquisicion", detalle=f"{fase}: {type(exc).__name__}: {exc}")
    finally:
        if temporal is not None:
            temporal.unlink(missing_ok=True)
    return info

adquisiciones = [adquirir(f) for f in FUENTES]
tabla_adquisiciones = pd.DataFrame(adquisiciones)
display(tabla_adquisiciones[["id_fuente", "estado", "bytes", "miembros", "ruta_relativa", "detalle"]])

,id_fuente,estado,bytes,miembros,ruta_relativa,detalle
0,REG_2026_PRES_1V,existente_validado,16348265.0,7.0,datos/brutos/registraduria/2026/MMV_Presidente...,
1,REG_2026_PRES_2V,existente_validado,6428218.0,7.0,datos/brutos/presidencia/2026/REG_2026_PRES_2V...,
2,REG_2026_CONGRESO,existente_validado,521530784.0,8.0,datos/brutos/congreso/2026/REG_2026_CONGRESO_M...,
3,REG_2023_TERR,sin_url_resuelta,NaN,NaN,,Selector departamental: URL exacta de Santande...
4,REG_2019_TERR,sin_url_resuelta,NaN,NaN,,Selector departamental: URL exacta de Santande...
5,REG_2022_PRES_1V,existente_validado,5064481.0,1.0,datos/brutos/presidencia/2022/REG_2022_PRES_1V...,
6,REG_2022_PRES_2V,existente_validado,2933014.0,1.0,datos/brutos/presidencia/2022/REG_2022_PRES_2V...,
7,REG_2022_CONGRESO,error_adquisicion,NaN,NaN,,integridad: ValueError: la respuesta es una pá...
8,REG_CENSO_HIST,sin_url_resuelta,NaN,NaN,,Consulta interactiva; exportar años y eleccion...
9,DANE_MUN_EDAD_2018_2042,existente_validado,131609579.0,27.0,datos/brutos/poblacion/serie/DANE_MUN_EDAD_201...,


## 5. Inventario físico

El inventario registra los nombres, tamaños y CRC de los archivos dentro de cada ZIP, y las hojas de los XLSX. Los originales se conservan sin cambios. Las columnas de las tablas nuevas se revisan en la sección 7.

In [5]:
def inventario_archivo(f: dict, d: dict) -> list[dict]:
    if d["estado"] not in {"existente_validado", "descargado_validado", "manual_validado"}:
        return []
    ruta = RAIZ / d["ruta_relativa"]
    registros = []
    if f["extension"] in {"zip", "xlsx"}:
        with zipfile.ZipFile(ruta) as z:
            for m in z.infolist():
                if m.is_dir():
                    continue
                registros.append({"id_fuente": f["id_fuente"], "miembro": m.filename,
                                  "bytes_descomprimidos": m.file_size, "crc32": f"{m.CRC:08x}"})
            if f["extension"] == "xlsx":
                raiz_xml = ElementTree.fromstring(z.read("xl/workbook.xml"))
                ns = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
                hojas = [x.attrib["name"] for x in raiz_xml.findall(".//m:sheet", ns)]
                for r in registros:
                    r["hojas_xlsx"] = " | ".join(hojas)
    else:
        with ruta.open("rb") as stream:
            muestra = stream.read(64 * 1024)
        codec = "utf-8-sig" if not muestra.startswith(b"\xff\xfe") else "utf-16"
        try:
            texto = muestra.decode(codec)
        except UnicodeDecodeError:
            codec, texto = "cp1252", muestra.decode("cp1252")
        primera = texto.splitlines()[0][:500] if texto.splitlines() else ""
        registros.append({"id_fuente": f["id_fuente"], "miembro": ruta.name,
                          "bytes_descomprimidos": ruta.stat().st_size,
                          "codificacion_muestra": codec, "primera_linea_muestra": primera})
    return registros

inventario_miembros = []
for f, d in zip(FUENTES, adquisiciones):
    inventario_miembros.extend(inventario_archivo(f, d))
tabla_miembros = pd.DataFrame(inventario_miembros)
print("Elementos físicos inventariados:", len(tabla_miembros))
display(tabla_miembros.head(25))

Elementos físicos inventariados: 69


,id_fuente,miembro,bytes_descomprimidos,crc32,hojas_xlsx
0,REG_2026_PRES_1V,MMV_Presidente1V_2026/MMV_Presidente1V_2026/AR...,2143812,42dbfb6d,NaN
1,REG_2026_PRES_1V,MMV_Presidente1V_2026/MMV_Presidente1V_2026/Es...,173191,b46ad489,NaN
2,REG_2026_PRES_1V,MMV_Presidente1V_2026/MMV_Presidente1V_2026/HA...,1452,2f51f37a,NaN
3,REG_2026_PRES_1V,MMV_Presidente1V_2026/MMV_Presidente1V_2026/HA...,1428,9ee62991,NaN
4,REG_2026_PRES_1V,MMV_Presidente1V_2026/MMV_Presidente1V_2026/No...,2118630,5919efa9,NaN
5,REG_2026_PRES_1V,MMV_Presidente1V_2026/MMV_Presidente1V_2026/PR...,59180240,c6b9158c,NaN
6,REG_2026_PRES_1V,MMV_Presidente1V_2026/MMV_Presidente1V_2026/_f...,105425280,8c254e38,NaN
7,REG_2026_PRES_2V,MMV_Presidente2V_2026/ArchivosBasicosAuditores...,313345,ebef5df9,NaN
8,REG_2026_PRES_2V,MMV_Presidente2V_2026/Estructuras Basicas (180...,173191,b46ad489,NaN
9,REG_2026_PRES_2V,MMV_Presidente2V_2026/HASH_ficheros_MMV_4_MMV_...,1428,2420836f,NaN


## 6. Presidencia 2026, primera vuelta

El adaptador consulta el PDF de estructuras del ZIP, verifica las huellas SHA-256 e identifica Barrancabermeja en DIVIPOL. Lee 38 bytes por fila de preconteo y 12 campos separados por punto y coma en escrutinio. Usa los catálogos disponibles y compara los resultados con las novedades por mesa. Si no se obtiene el ZIP, la fuente queda pendiente.

In [6]:
PILOTO_DISPONIBLE = next(
    (d["estado"] in {"existente_validado", "descargado_validado", "manual_validado"}
     for d in adquisiciones if d["id_fuente"] == "REG_2026_PRES_1V"), False)
qa_piloto_generado = {}
salidas_piloto = {}
verificaciones = pd.DataFrame([{
    "id_fuente": f["id_fuente"], "estado_red": "no_sondeado",
    "detalle": "La descarga completa registra la respuesta HTTP; no se envía un sondeo adicional"
} for f in FUENTES if f["url_directa"]])
resultados_descarga = tabla_adquisiciones.copy()
print("Piloto presente e íntegro:", PILOTO_DISPONIBLE)

Piloto presente e íntegro: True


In [7]:
if PILOTO_DISPONIBLE:
    from collections import Counter
    from pathlib import PurePosixPath

    ZIP_PILOTO = RAIZ / next(d["ruta_relativa"] for d in adquisiciones if d["id_fuente"] == "REG_2026_PRES_1V")
    if not ZIP_PILOTO.is_file():
        raise FileNotFoundError(f"No está el ZIP del piloto en {ZIP_PILOTO}. Ejecuta la adquisición de la sección 4.")

    def miembro_unico(archivo: zipfile.ZipFile, nombre: str) -> str:
        """Localiza un miembro por su nombre final, sin depender de carpetas anidadas."""
        coincidencias = [x for x in archivo.namelist() if PurePosixPath(x).name == nombre]
        if len(coincidencias) != 1:
            raise ValueError(f"Se esperaba exactamente un {nombre!r}; encontrados: {coincidencias}")
        return coincidencias[0]

    def miembro_por_prefijo(archivo: zipfile.ZipFile, prefijo: str, extension: str) -> str:
        coincidencias = [x for x in archivo.namelist()
                         if PurePosixPath(x).name.startswith(prefijo) and x.endswith(extension)]
        if len(coincidencias) != 1:
            raise ValueError(f"Se esperaba un {prefijo}*{extension}; encontrados: {coincidencias}")
        return coincidencias[0]

    def sha256_bytes(datos: bytes) -> str:
        return hashlib.sha256(datos).hexdigest()

    def leer_fijo(datos: bytes, campos: list[tuple[str, int]], nombre: str) -> pd.DataFrame:
        """Valida todos los registros y conserva los códigos como texto (incluidos ceros)."""
        ancho = sum(ancho for _, ancho in campos)
        filas = []
        for numero, linea in enumerate(datos.splitlines(), start=1):
            if len(linea) != ancho:
                raise ValueError(f"{nombre}: registro {numero} de {len(linea)} bytes; se esperaban {ancho}")
            cursor, fila = 0, {}
            for campo, longitud in campos:
                fila[campo] = linea[cursor:cursor + longitud].decode("cp1252").strip()
                cursor += longitud
            filas.append(fila)
        return pd.DataFrame(filas, columns=[nombre for nombre, _ in campos])

    def leer_hash_publicado(texto: str) -> str:
        encontrado = re.search(r"SHA-256\s*:\s*([0-9a-fA-F]{64})", texto)
        if not encontrado:
            raise ValueError("Falta SHA-256 en el manifiesto oficial")
        return encontrado.group(1).lower()

    DIVIPOL_CAMPOS = [
        ("cod_depto", 2), ("cod_municipio", 3), ("cod_zona", 2), ("cod_puesto", 2),
        ("departamento", 12), ("municipio", 30), ("puesto", 40), ("indicador_puesto", 1),
        ("potencial_hombres", 8), ("potencial_mujeres", 8), ("mesas_declaradas", 6),
        ("cod_comuna_divipol", 2), ("comuna_divipol", 30),
    ]
    CANDIDATOS_CAMPOS = [
        ("cod_corporacion", 3), ("cod_circunscripcion", 1), ("cod_depto", 2),
        ("cod_municipio", 3), ("cod_comuna", 2), ("cod_partido", 5),
        ("cod_candidato", 3), ("tipo_preferente", 1), ("nombre", 50),
        ("apellido", 50), ("cedula", 15), ("genero", 1), ("sorteo", 2),
    ]

    with zipfile.ZipFile(ZIP_PILOTO) as z:
        PRE = miembro_unico(z, "PRE1v_MMV_9999_PRECONTEO.txt")
        ESC = miembro_unico(z, "_ficheros_MMV_4_MMV_9999_ESCRUTINIO.csv")
        hpre = miembro_unico(z, "HASH_PRE1v_MMV_9999_PRECONTEO.txt")
        hesc = miembro_unico(z, "HASH_ficheros_MMV_4_MMV_9999_ESCRUTINIO.txt")
        documento = miembro_unico(z, "Estructuras Basicas (1808).pdf")
        basicos_zip = miembro_unico(z, "ARCHIVOSBASICOS_AUDITORES_PRESIDENTE_2026_V3.zip")
        novedades_zip = miembro_unico(z, "Novedad20260603.zip")
        huellas = {}
        for fuente, manifiesto_hash in [(PRE, hpre), (ESC, hesc)]:
            esperado = leer_hash_publicado(z.read(manifiesto_hash).decode("utf-16"))
            with z.open(fuente) as stream:
                calculado = hashlib.file_digest(stream, "sha256").hexdigest()
            if calculado != esperado:
                raise AssertionError(f"SHA-256 oficial no coincide: {PurePosixPath(fuente).name}")
            huellas[PurePosixPath(fuente).name] = calculado
        huella_documento = sha256_bytes(z.read(documento))
        basicos_bytes = z.read(basicos_zip)
        novedades_bytes = z.read(novedades_zip)
        inventario = [{"miembro": x.filename, "bytes_descomprimidos": x.file_size,
                       "crc32_zip": f"{x.CRC:08x}"} for x in z.infolist() if not x.is_dir()]

    with zipfile.ZipFile(io.BytesIO(basicos_bytes)) as basicos:
        n_div = miembro_por_prefijo(basicos, "DIVIPOL_", ".TXT")
        n_can = miembro_por_prefijo(basicos, "CANDIDATOS_", ".TXT")
        n_par = miembro_por_prefijo(basicos, "PARTIDOS_", ".TXT")
        hash_basicos = basicos.read(miembro_unico(basicos, "HASH.txt")).decode("utf-16")
        fichas_hash = re.findall(
            r"Filename\s*:\s*([^\r\n]+).*?SHA-256\s*:\s*([0-9a-fA-F]{64})",
            hash_basicos, flags=re.DOTALL,
        )
        if len(fichas_hash) != len([m for m in basicos.namelist() if m.endswith(".TXT") and not m.endswith("HASH.txt")]):
            raise AssertionError("No se pudo cotejar cada archivo básico con su manifiesto SHA-256")
        for nombre, huella in fichas_hash:
            encontrado = miembro_unico(basicos, nombre.strip())
            if sha256_bytes(basicos.read(encontrado)) != huella.lower():
                raise AssertionError(f"SHA-256 del archivo básico incorrecto: {nombre}")
        divipol = leer_fijo(basicos.read(n_div), DIVIPOL_CAMPOS, "DIVIPOL")
        candidatos = leer_fijo(basicos.read(n_can), CANDIDATOS_CAMPOS, "CANDIDATOS")
        partidos_raw = basicos.read(n_par).splitlines()

    # PARTIDOS tiene un carácter adicional observado después de los 205 documentados.
    # Solo se interpretan los 205 bytes descritos en el PDF; se conserva el resto para auditoría.
    if any(len(x) != 206 for x in partidos_raw):
        raise ValueError("PARTIDOS cambió su longitud observada (206 bytes por registro)")
    partidos = {x[:5].decode("cp1252"): x[5:205].decode("cp1252").strip()
                for x in partidos_raw}
    if len(partidos) != len(partidos_raw):
        raise AssertionError("Códigos de partido duplicados")

    geocampos = ["cod_depto", "cod_municipio", "cod_zona", "cod_puesto"]
    if divipol.duplicated(geocampos).any():
        raise AssertionError("Llaves departamento/municipio/zona/puesto duplicadas en DIVIPOL")
    local = divipol.loc[
        divipol["departamento"].str.upper().eq("SANTANDER") &
        divipol["municipio"].str.upper().eq("BARRANCABERMEJA")
    ].copy()
    codigos = local[["cod_depto", "cod_municipio"]].drop_duplicates()
    if len(local) == 0 or len(codigos) != 1:
        raise AssertionError("DIVIPOL no identifica de forma única a Barrancabermeja, Santander")
    COD_DEPTO, COD_MUNICIPIO = codigos.iloc[0].tolist()
    puestos = {tuple(fila[c] for c in geocampos): fila for fila in local.to_dict("records")}
    claves_candidato = ["cod_partido", "cod_candidato"]
    if candidatos.duplicated(claves_candidato).any():
        raise AssertionError("La combinación partido/candidato no es única en CANDIDATOS")
    candidaturas = {(r["cod_partido"], r["cod_candidato"]): r
                    for r in candidatos.to_dict("records")}
    if any(r["cod_partido"] not in partidos for r in candidaturas.values()):
        raise AssertionError("Candidato con partido ausente del catálogo básico")

    manifiesto = pd.DataFrame([{
        "id_fuente": "REG_2026_PRES_1V",
        "ruta_relativa": ZIP_PILOTO.relative_to(RAIZ).as_posix(),
        "bytes": ZIP_PILOTO.stat().st_size,
        "sha256": sha256_archivo(ZIP_PILOTO),
        "miembros_zip": len(inventario),
        "sha256_pdf_estructura": huella_documento,
    }])
    display(pd.DataFrame(inventario))
    print("Huellas oficiales verificadas:", len(huellas), "archivos de votos y",
          len(fichas_hash), "archivos básicos.")
    print("Código electoral Barrancabermeja:", COD_DEPTO, COD_MUNICIPIO,
          "| Puestos DIVIPOL:", len(local),
          "| Mesas DIVIPOL:", int(local.mesas_declaradas.astype(int).sum()))


else:
    print('Piloto 2026 1V pendiente: el resto de la auditoría sigue disponible.')

,miembro,bytes_descomprimidos,crc32_zip
0,MMV_Presidente1V_2026/MMV_Presidente1V_2026/AR...,2143812,42dbfb6d
1,MMV_Presidente1V_2026/MMV_Presidente1V_2026/Es...,173191,b46ad489
2,MMV_Presidente1V_2026/MMV_Presidente1V_2026/HA...,1452,2f51f37a
3,MMV_Presidente1V_2026/MMV_Presidente1V_2026/HA...,1428,9ee62991
4,MMV_Presidente1V_2026/MMV_Presidente1V_2026/No...,2118630,5919efa9
5,MMV_Presidente1V_2026/MMV_Presidente1V_2026/PR...,59180240,c6b9158c
6,MMV_Presidente1V_2026/MMV_Presidente1V_2026/_f...,105425280,8c254e38


Huellas oficiales verificadas: 2 archivos de votos y 6 archivos básicos.
Código electoral Barrancabermeja: 27 019 | Puestos DIVIPOL: 67 | Mesas DIVIPOL: 566


In [8]:
if PILOTO_DISPONIBLE:
    PRE_CAMPOS = [
        ("cod_depto", 2), ("cod_municipio", 3), ("cod_zona", 2), ("cod_puesto", 2),
        ("mesa", 6), ("cod_jal", 2), ("comunicado", 4), ("cod_circunscripcion", 1),
        ("cod_partido", 5), ("cod_candidato", 3), ("votos", 8),
    ]
    ESC_CAMPOS = [
        "valor_fijo", "cod_depto", "cod_municipio", "cod_zona", "cod_puesto",
        "mesa", "cod_comuna_escrutinio", "cod_corporacion", "cod_circunscripcion",
        "cod_partido", "cod_candidato", "votos",
    ]
    ESPECIALES = {"996", "997", "998"}  # El ZIP no aporta nombres para estos códigos.

    def normalizar_zona_escrutinio(zona: str) -> str:
        # El PDF especifica 3 posiciones en escrutinio y 2 en DIVIPOL.
        if len(zona) != 3 or not zona.startswith("0"):
            raise ValueError(f"Zona de escrutinio sin equivalencia documentada con DIVIPOL: {zona!r}")
        return zona[1:]

    def interpretar_registro(linea: bytes, tipo: str, numero: int) -> dict:
        """Interpreta una fila según el PDF interno, con validación de anchura y códigos."""
        registro = linea.rstrip(b"\r\n")
        if tipo == "preconteo":
            if len(registro) != 38:
                raise ValueError(f"{tipo}, línea {numero}: {len(registro)} bytes en vez de 38")
            fila, inicio = {}, 0
            for nombre, ancho in PRE_CAMPOS:
                fila[nombre] = registro[inicio:inicio + ancho].decode("ascii")
                inicio += ancho
            if fila["comunicado"] != "9999":
                raise ValueError(f"{tipo}, línea {numero}: comunicado inesperado")
        else:
            campos = registro.decode("utf-8").split(";")
            if len(campos) != 13 or campos[-1] != "":
                raise ValueError(f"{tipo}, línea {numero}: se esperaban 12 campos y ; final")
            fila = dict(zip(ESC_CAMPOS, campos[:-1]))
            if fila["valor_fijo"] != "9999" or fila["cod_corporacion"] != "001":
                raise ValueError(f"{tipo}, línea {numero}: identificador o corporación inesperados")
        for nombre in ("cod_depto", "cod_municipio", "mesa", "cod_partido", "cod_candidato", "votos"):
            if not fila[nombre].isdigit():
                raise ValueError(f"{tipo}, línea {numero}: campo numérico {nombre} inválido")
        return fila

    def extraer_local(nombre_archivo: str, tipo: str) -> tuple[pd.DataFrame, dict]:
        registros = []
        total = 0
        con_zip = zipfile.ZipFile(ZIP_PILOTO)
        try:
            with con_zip.open(nombre_archivo) as stream:
                for total, linea in enumerate(stream, start=1):
                    fila = interpretar_registro(linea, tipo, total)
                    if (fila["cod_depto"], fila["cod_municipio"]) != (COD_DEPTO, COD_MUNICIPIO):
                        continue
                    if tipo == "escrutinio":
                        zona_original = fila["cod_zona"]
                        fila["cod_zona"] = normalizar_zona_escrutinio(zona_original)
                        fila["cod_partido"] = fila["cod_partido"].zfill(5)
                    geo = tuple(fila[c] for c in geocampos)
                    if geo not in puestos:
                        raise AssertionError(f"{tipo}, línea {total}: puesto local ausente en DIVIPOL: {geo}")
                    candidato = (fila["cod_partido"], fila["cod_candidato"])
                    if candidato in candidaturas:
                        c = candidaturas[candidato]
                        nombre_candidato = f'{c["nombre"]} {c["apellido"]}'.strip()
                        partido = partidos[c["cod_partido"]]
                        clase = "candidatura_catalogada"
                    elif fila["cod_partido"] == "00000" and fila["cod_candidato"] in ESPECIALES:
                        nombre_candidato, partido, clase = "", "", "opcion_especial_sin_etiqueta"
                    else:
                        raise AssertionError(f"{tipo}, línea {total}: opción sin equivalencia: {candidato}")
                    puesto = puestos[geo]
                    registros.append({
                        "id_fuente": "REG_2026_PRES_1V", "tipo_resultado": tipo,
                        "cod_depto": fila["cod_depto"], "departamento": puesto["departamento"],
                        "cod_municipio": fila["cod_municipio"], "municipio": puesto["municipio"],
                        "cod_zona": fila["cod_zona"], "cod_puesto": fila["cod_puesto"],
                        "puesto": puesto["puesto"], "mesa": fila["mesa"],
                        "cod_comuna_divipol": puesto["cod_comuna_divipol"],
                        "comuna_divipol": puesto["comuna_divipol"],
                        "cod_comuna_escrutinio": fila.get("cod_comuna_escrutinio", ""),
                        "cod_partido": fila["cod_partido"], "partido": partido,
                        "cod_candidato": fila["cod_candidato"], "candidato": nombre_candidato,
                        "clase_opcion": clase, "votos": int(fila["votos"]),
                        "archivo_origen": PurePosixPath(nombre_archivo).name,
                    })
        finally:
            con_zip.close()
        datos = pd.DataFrame(registros)
        if datos.empty:
            raise AssertionError(f"{tipo}: sin filas del municipio {COD_DEPTO}/{COD_MUNICIPIO}")
        clave = ["cod_depto", "cod_municipio", "cod_zona", "cod_puesto", "mesa",
                 "cod_partido", "cod_candidato"]
        if datos.duplicated(clave).any():
            raise AssertionError(f"{tipo}: llave mesa/partido/candidato duplicada")
        return datos, {"archivo": nombre_archivo, "filas_nacionales": total,
                       "filas_locales": len(datos),
                       "mesas_locales": int(datos[["cod_zona", "cod_puesto", "mesa"]].drop_duplicates().shape[0]),
                       "votos_locales": int(datos.votos.sum())}

    preconteo, resumen_pre = extraer_local(PRE, "preconteo")
    escrutinio, resumen_esc = extraer_local(ESC, "escrutinio")
    display(pd.DataFrame([{"tipo": "preconteo", **resumen_pre},
                          {"tipo": "escrutinio", **resumen_esc}]))


else:
    print('Piloto 2026 1V pendiente: el resto de la auditoría sigue disponible.')

,tipo,archivo,filas_nacionales,filas_locales,mesas_locales,votos_locales
0,preconteo,MMV_Presidente1V_2026/MMV_Presidente1V_2026/PR...,1479506,8941,566,121378
1,escrutinio,MMV_Presidente1V_2026/MMV_Presidente1V_2026/_f...,1952320,9056,566,121683


In [9]:
if PILOTO_DISPONIBLE:
    with zipfile.ZipFile(io.BytesIO(novedades_bytes)) as nov_zip:
        nov_csv = miembro_unico(nov_zip, "Novedad20260603.csv")
        nov_hash = miembro_unico(nov_zip, "HuellaHash.txt")
        huella_novedades = nov_zip.read(nov_hash).decode("ascii").strip().lower()
        with nov_zip.open(nov_csv) as stream:
            if hashlib.file_digest(stream, "sha256").hexdigest() != huella_novedades:
                raise AssertionError("Novedades: SHA-256 no coincide con HuellaHash.txt")
        nov_rows, nov_nacionales = [], 0
        with nov_zip.open(nov_csv) as stream:
            reader = csv.DictReader(io.TextIOWrapper(stream, encoding="utf-8-sig", newline=""))
            obligatorias = {"departamentoCodigo", "municipioCodigo", "zonaNombre", "puestoCodigo",
                            "mesaNumero", "totalVotos", "potencial", "sufragantesSegunE11"}
            if not obligatorias.issubset(reader.fieldnames or []):
                raise ValueError("Novedades: faltan campos requeridos")
            for nov_nacionales, fila in enumerate(reader, start=1):
                if (fila["departamentoCodigo"], fila["municipioCodigo"]) == (COD_DEPTO, COD_MUNICIPIO):
                    if fila["municipioNombre"].strip().upper() != "BARRANCABERMEJA":
                        raise AssertionError("Novedades: código municipal asociado con nombre discordante")
                    fila["mesa"] = fila["mesaNumero"].zfill(6)
                    fila["cod_zona"] = fila["zonaNombre"].zfill(2)
                    fila["cod_puesto"] = fila["puestoCodigo"].zfill(2)
                    nov_rows.append(fila)
    novedades = pd.DataFrame(nov_rows)
    MESAS = ["cod_zona", "cod_puesto", "mesa"]
    if novedades.empty or novedades.duplicated(MESAS).any():
        raise AssertionError("Novedades locales ausentes o con mesas duplicadas")
    if not novedades["totalVotos"].str.fullmatch(r"\d+").all():
        raise ValueError("Novedades: totalVotos no numérico")

    votos_pre = preconteo.groupby(MESAS, as_index=False).agg(
        votos_preconteo=("votos", "sum"), filas_preconteo=("votos", "size"))
    votos_esc = escrutinio.groupby(MESAS, as_index=False).agg(
        votos_escrutinio=("votos", "sum"), filas_escrutinio=("votos", "size"))
    control_mesas = votos_esc.merge(votos_pre, on=MESAS, how="outer", validate="one_to_one",
                                    indicator="_cruce_pre_escr")
    control_mesas = control_mesas.merge(
        novedades[MESAS + ["totalVotos", "potencial", "sufragantesSegunE11"]],
        on=MESAS, how="outer", validate="one_to_one", indicator="_cruce_novedades")
    if control_mesas[["votos_preconteo", "votos_escrutinio", "totalVotos"]].isna().any().any():
        raise AssertionError("Alguna mesa no está en las tres fuentes")
    control_mesas["totalVotos"] = control_mesas["totalVotos"].astype(int)
    control_mesas["diferencia_escrutinio_preconteo"] = (
        control_mesas["votos_escrutinio"] - control_mesas["votos_preconteo"])
    control_mesas["cuadra_con_novedades"] = (
        control_mesas["votos_escrutinio"] == control_mesas["totalVotos"])
    if not control_mesas.cuadra_con_novedades.all():
        raise AssertionError("Hay mesas cuyo escrutinio no coincide con el totalVotos de novedades")
    if len(control_mesas) != int(local.mesas_declaradas.astype(int).sum()):
        raise AssertionError("Cantidad de mesas no coincide con DIVIPOL")
    if not nov_nacionales:
        raise AssertionError("Archivo de novedades vacío")
    mesas_por_puesto = control_mesas.groupby(["cod_zona", "cod_puesto"]).size().to_dict()
    desajustes_puestos = [
        (r["cod_zona"], r["cod_puesto"], int(r["mesas_declaradas"]),
         mesas_por_puesto.get((r["cod_zona"], r["cod_puesto"]), 0))
        for r in local.to_dict("records")
        if int(r["mesas_declaradas"]) !=
        mesas_por_puesto.get((r["cod_zona"], r["cod_puesto"]), 0)
    ]
    if desajustes_puestos:
        raise AssertionError(f"Mesas por puesto distintas de DIVIPOL: {desajustes_puestos[:5]}")

    qa = {
        "zip_sha256": sha256_archivo(ZIP_PILOTO),
        "pdf_estructura_sha256": huella_documento,
        "sha256_oficial_preconteo": huellas[PurePosixPath(PRE).name],
        "sha256_oficial_escrutinio": huellas[PurePosixPath(ESC).name],
        "sha256_oficial_novedades": huella_novedades,
        "archivos_basicos_con_hash_verificado": len(fichas_hash),
        "codigo_electoral": {"departamento": COD_DEPTO, "municipio": COD_MUNICIPIO},
        "puestos_divipol": len(local),
        "puestos_con_mesas_conciliadas_divipol": len(local) - len(desajustes_puestos),
        "mesas_divipol": int(local.mesas_declaradas.astype(int).sum()),
        "preconteo": resumen_pre,
        "escrutinio": resumen_esc,
        "novedades": {"filas_nacionales": nov_nacionales, "mesas_locales": len(novedades),
                      "total_votos_local": int(novedades.totalVotos.astype(int).sum()),
                      "potencial_local": int(novedades.potencial.astype(int).sum()),
                      "sufragantes_E11_local": int(novedades.sufragantesSegunE11.astype(int).sum())},
        "mesas_con_diferencia_preconteo_escrutinio": int(control_mesas.diferencia_escrutinio_preconteo.ne(0).sum()),
        "diferencia_neta_votos_escrutinio_menos_preconteo": int(control_mesas.diferencia_escrutinio_preconteo.sum()),
        "mesas_escrutinio_vs_novedades_conciliadas": int(control_mesas.cuadra_con_novedades.sum()),
        "codigos_especiales_sin_etiqueta": sorted(ESPECIALES),
        "criterio_avance_piloto": "aprobado",
        "limite": "Piloto presidencial 2026 1V; las otras elecciones del catálogo siguen pendientes.",
    }

    destino = RAIZ / "datos" / "intermedios" / "registraduria" / "2026"
    destino.mkdir(parents=True, exist_ok=True)
    salidas = {
        "preconteo": (preconteo, destino / "presidencia_2026_1v_preconteo_barrancabermeja.csv"),
        "escrutinio": (escrutinio, destino / "presidencia_2026_1v_escrutinio_barrancabermeja.csv"),
        "novedades": (novedades, destino / "presidencia_2026_1v_novedades_barrancabermeja.csv"),
        "control_mesas": (control_mesas, destino / "presidencia_2026_1v_control_mesas.csv"),
        "divipol_local": (local, destino / "presidencia_2026_1v_divipol_barrancabermeja.csv"),
    }
    for etiqueta, (datos, ruta) in salidas.items():
        temporal = ruta.with_suffix(ruta.suffix + ".part")
        try:
            datos.to_csv(temporal, index=False, encoding="utf-8-sig")
            temporal.replace(ruta)
        finally:
            temporal.unlink(missing_ok=True)
    qa["salidas"] = {etiqueta: ruta.relative_to(RAIZ).as_posix()
                     for etiqueta, (_, ruta) in salidas.items()}
    catalogo.to_csv(AUDITORIA / "catalogo_fuentes.csv", index=False, encoding="utf-8-sig")
    verificaciones.to_csv(AUDITORIA / "verificacion_urls.csv", index=False, encoding="utf-8-sig")
    resultados_descarga.to_csv(AUDITORIA / "resultados_descarga.csv", index=False, encoding="utf-8-sig")
    manifiesto.to_csv(AUDITORIA / "manifiesto_brutos.csv", index=False, encoding="utf-8-sig")
    (AUDITORIA / "inventario_miembros.json").write_text(
        json.dumps(inventario, ensure_ascii=False, indent=2), encoding="utf-8")
    (AUDITORIA / "control_calidad.json").write_text(
        json.dumps(qa, ensure_ascii=False, indent=2), encoding="utf-8")
    print("APROBADO: piloto 2026 primera vuelta listo para análisis descriptivo.")
    print("Mesas:", len(control_mesas), "| Preconteo:", len(preconteo), "filas,",
          int(preconteo.votos.sum()), "votos | Escrutinio:", len(escrutinio), "filas,",
          int(escrutinio.votos.sum()), "votos")
    print("Diferencias preconteo/escrutinio:", qa["mesas_con_diferencia_preconteo_escrutinio"],
          "mesas | Diferencia neta:", qa["diferencia_neta_votos_escrutinio_menos_preconteo"])
    print("Escrutinio y novedades conciliados:", qa["mesas_escrutinio_vs_novedades_conciliadas"],
          "de", len(control_mesas), "mesas")
    display(pd.DataFrame([{"archivo": k, "ruta_relativa": v} for k, v in qa["salidas"].items()]))
    display(preconteo[["cod_zona", "cod_puesto", "mesa", "cod_candidato", "candidato", "votos"]].head(5))

    qa_piloto_generado = qa
    salidas_piloto = qa["salidas"]
    estable = RAIZ / "docs" / "fuentes" / "auditorias" / "piloto_2026_1v"
    estable.mkdir(parents=True, exist_ok=True)
    (estable / "control_calidad.json").write_text(
        json.dumps(qa, ensure_ascii=False, indent=2), encoding="utf-8")
    (estable / "sha256_derivados.json").write_text(
        json.dumps({v: sha256_archivo(RAIZ / v) for v in qa["salidas"].values()},
                   ensure_ascii=False, indent=2), encoding="utf-8")

else:
    print('Piloto 2026 1V pendiente: el resto de la auditoría sigue disponible.')

APROBADO: piloto 2026 primera vuelta listo para análisis descriptivo.
Mesas: 566 | Preconteo: 8941 filas, 121378 votos | Escrutinio: 9056 filas, 121683 votos
Diferencias preconteo/escrutinio: 33 mesas | Diferencia neta: 305
Escrutinio y novedades conciliados: 566 de 566 mesas


,archivo,ruta_relativa
0,preconteo,datos/intermedios/registraduria/2026/presidenc...
1,escrutinio,datos/intermedios/registraduria/2026/presidenc...
2,novedades,datos/intermedios/registraduria/2026/presidenc...
3,control_mesas,datos/intermedios/registraduria/2026/presidenc...
4,divipol_local,datos/intermedios/registraduria/2026/presidenc...


,cod_zona,cod_puesto,mesa,cod_candidato,candidato,votos
0,01,01,000001,011,PALOMA VALENCIA LASERNA,14
1,01,01,000001,012,SERGIO FAJARDO VALDERRAMA,2
2,01,01,000001,006,MIGUEL URIBE LONDOÑO,0
3,01,01,000001,010,GUSTAVO MATAMOROS CAMACHO,0
4,01,01,000001,008,ROY LEONARDO BARRERAS MONTEALEGRE,0


### 6.1 Presidencia 2026, segunda vuelta

El adaptador comprueba que el PDF de estructura coincida con el del piloto y verifica su SHA-256, las huellas publicadas y la consistencia de escrutinio, DIVIPOL y novedades por mesa. Si alguna validación falla, la fuente queda pendiente y se registra el motivo. Las salidas se generan desde el original de segunda vuelta, sin usar datos de la primera.

In [10]:
from collections import Counter

def procesar_presidencia_2026_2v(adquisicion: dict) -> dict:
    """Interpreta 2V solo si la estructura es idéntica a la del PDF auditado en 1V."""
    ruta = RAIZ / adquisicion["ruta_relativa"]
    with zipfile.ZipFile(ruta) as z:
        pre = miembro_unico(z, "PRE2v_MMV_9999_PRECONTEO.txt")
        esc = miembro_unico(z, "_ficheros_MMV_4_MMV_9999_ESCRUTINIO.csv")
        hpre = miembro_unico(z, "HASH_PRE2v_MMV_9999_PRECONTEO.txt")
        hesc = miembro_unico(z, "HASH_ficheros_MMV_4_MMV_9999_ESCRUTINIO.txt")
        pdf = miembro_unico(z, "Estructuras Basicas (1808).pdf")
        if sha256_bytes(z.read(pdf)) != huella_documento:
            raise ValueError("La estructura PDF de 2V difiere del esquema auditado en 1V")
        basicos_nombres = [n for n in z.namelist() if re.search(r"(?i)archivosbasicos.*\.zip$", n)]
        nov_nombres = [n for n in z.namelist() if re.search(r"(?i)novedad.*\.zip$", n)]
        if len(basicos_nombres) != 1 or len(nov_nombres) != 1:
            raise ValueError("No se identifican de forma única básicos y novedades 2V")
        huellas_2v = {}
        for archivo, manifiesto_hash in ((pre, hpre), (esc, hesc)):
            texto = z.read(manifiesto_hash).decode("utf-16")
            esperada = leer_hash_publicado(texto)
            with z.open(archivo) as stream:
                calculada = hashlib.file_digest(stream, "sha256").hexdigest()
            if calculada != esperada:
                raise AssertionError("Huella oficial discordante: " + archivo)
            huellas_2v[PurePosixPath(archivo).name] = calculada
        basicos_bytes = z.read(basicos_nombres[0])
        novedades_bytes_2v = z.read(nov_nombres[0])

        with zipfile.ZipFile(io.BytesIO(basicos_bytes)) as b:
            hash_basicos = b.read(miembro_unico(b, "HASH.txt")).decode("utf-16")
            entradas = re.findall(
                r"Filename\s*:\s*([^\r\n]+).*?SHA-256\s*:\s*([0-9a-fA-F]{64})",
                hash_basicos, flags=re.DOTALL,
            )
            archivos_txt = [n for n in b.namelist() if n.upper().endswith(".TXT") and
                            PurePosixPath(n).name.upper() != "HASH.TXT"]
            if len(entradas) != len(archivos_txt):
                raise AssertionError("Archivos básicos sin cobertura completa en HASH.txt")
            for nombre, huella in entradas:
                if sha256_bytes(b.read(miembro_unico(b, nombre.strip()))) != huella.lower():
                    raise AssertionError("Huella discordante en básicos: " + nombre)
            div = leer_fijo(b.read(miembro_por_prefijo(b, "DIVIPOL_", ".TXT")),
                            DIVIPOL_CAMPOS, "DIVIPOL_2V")
            can = leer_fijo(b.read(miembro_por_prefijo(b, "CANDIDATOS_", ".TXT")),
                            CANDIDATOS_CAMPOS, "CANDIDATOS_2V")
            part_filas = b.read(miembro_por_prefijo(b, "PARTIDOS_", ".TXT")).splitlines()

        if div.duplicated(["cod_depto", "cod_municipio", "cod_zona", "cod_puesto"]).any():
            raise AssertionError("DIVIPOL de 2V contiene puestos duplicados")
        local_2v = div.loc[div.departamento.str.upper().eq("SANTANDER") &
                           div.municipio.str.upper().eq("BARRANCABERMEJA")].copy()
        codigos_2v = local_2v[["cod_depto", "cod_municipio"]].drop_duplicates()
        if len(codigos_2v) != 1 or tuple(codigos_2v.iloc[0]) != (COD_DEPTO, COD_MUNICIPIO):
            raise AssertionError("Municipio 2V no corresponde al código electoral auditado")
        puestos_2v = {tuple(r[c] for c in geocampos): r for r in local_2v.to_dict("records")}
        if can.duplicated(["cod_partido", "cod_candidato"]).any():
            raise AssertionError("Candidaturas duplicadas en básicos 2V")
        can_2v = {(r["cod_partido"], r["cod_candidato"]): r for r in can.to_dict("records")}
        if any(len(x) != 206 for x in part_filas):
            raise ValueError("PARTIDOS 2V no tiene los 206 bytes observados en 1V")
        partidos_2v = {x[:5].decode("cp1252"): x[5:205].decode("cp1252").strip()
                       for x in part_filas}
        if len(partidos_2v) != len(part_filas):
            raise AssertionError("Partidos duplicados en 2V")

        def extraer(archivo: str, tipo: str) -> tuple[pd.DataFrame, dict]:
            registros, nacionales = [], 0
            with z.open(archivo) as stream:
                for nacionales, linea in enumerate(stream, start=1):
                    fila = interpretar_registro(linea, tipo, nacionales)
                    if (fila["cod_depto"], fila["cod_municipio"]) != (COD_DEPTO, COD_MUNICIPIO):
                        continue
                    if tipo == "escrutinio":
                        fila["cod_zona"] = normalizar_zona_escrutinio(fila["cod_zona"])
                        fila["cod_partido"] = fila["cod_partido"].zfill(5)
                    geo = tuple(fila[c] for c in geocampos)
                    if geo not in puestos_2v:
                        raise AssertionError(f"{tipo}, línea {nacionales}: puesto ausente en DIVIPOL")
                    candidato = (fila["cod_partido"], fila["cod_candidato"])
                    if candidato in can_2v:
                        c = can_2v[candidato]
                        etiqueta = (c["nombre"] + " " + c["apellido"]).strip()
                        partido = partidos_2v[c["cod_partido"]]
                        clase = "candidatura_catalogada"
                    elif fila["cod_partido"] == "00000" and fila["cod_candidato"] in ESPECIALES:
                        etiqueta, partido, clase = "", "", "opcion_especial_sin_etiqueta"
                    else:
                        raise AssertionError(f"{tipo}, línea {nacionales}: opción no catalogada {candidato}")
                    puesto = puestos_2v[geo]
                    registros.append({
                        "id_fuente": "REG_2026_PRES_2V", "tipo_resultado": tipo,
                        "cod_depto": fila["cod_depto"], "departamento": puesto["departamento"],
                        "cod_municipio": fila["cod_municipio"], "municipio": puesto["municipio"],
                        "cod_zona": fila["cod_zona"], "cod_puesto": fila["cod_puesto"],
                        "puesto": puesto["puesto"], "mesa": fila["mesa"],
                        "cod_comuna_divipol": puesto["cod_comuna_divipol"],
                        "comuna_divipol": puesto["comuna_divipol"],
                        "cod_comuna_escrutinio": fila.get("cod_comuna_escrutinio", ""),
                        "cod_partido": fila["cod_partido"], "partido": partido,
                        "cod_candidato": fila["cod_candidato"], "candidato": etiqueta,
                        "clase_opcion": clase, "votos": int(fila["votos"]),
                        "archivo_origen": PurePosixPath(archivo).name,
                    })
            tabla = pd.DataFrame(registros)
            if tabla.empty or tabla.duplicated(MESAS + ["cod_partido", "cod_candidato"]).any():
                raise AssertionError("Votos locales vacíos o llaves duplicadas en " + tipo)
            resumen = {"filas_nacionales": nacionales, "filas_locales": len(tabla),
                       "mesas_locales": len(tabla[MESAS].drop_duplicates()),
                       "votos_locales": int(tabla.votos.sum())}
            return tabla, resumen

        pre_2v, resumen_pre_2v = extraer(pre, "preconteo")
        esc_2v, resumen_esc_2v = extraer(esc, "escrutinio")

    with zipfile.ZipFile(io.BytesIO(novedades_bytes_2v)) as novzip:
        novcsv = [n for n in novzip.namelist() if re.search(r"(?i)novedad.*\.csv$", n)]
        if len(novcsv) != 1:
            raise ValueError("Novedades CSV de 2V no identificado de forma única")
        huella_nov = novzip.read(miembro_unico(novzip, "HuellaHash.txt")).decode("ascii").strip().lower()
        with novzip.open(novcsv[0]) as stream:
            if hashlib.file_digest(stream, "sha256").hexdigest() != huella_nov:
                raise AssertionError("Novedades 2V: huella oficial discordante")
        novedades_rows = []
        with novzip.open(novcsv[0]) as stream:
            # El CSV de novedades publicado para 2V utiliza punto y coma;
            # en 1V el mismo proveedor había utilizado comas.
            reader = csv.DictReader(io.TextIOWrapper(stream, encoding="utf-8-sig", newline=""),
                                    delimiter=";")
            campos = {"departamentoCodigo", "municipioCodigo", "zonaNombre",
                      "puestoCodigo", "mesaNumero", "totalVotos"}
            if not campos.issubset(reader.fieldnames or []):
                raise ValueError("Novedades 2V: faltan " + str(sorted(campos - set(reader.fieldnames or []))) +
                                 "; columnas presentes: " + str(reader.fieldnames))
            for fila in reader:
                if (fila["departamentoCodigo"], fila["municipioCodigo"]) != (COD_DEPTO, COD_MUNICIPIO):
                    continue
                if fila.get("municipioNombre") and fila["municipioNombre"].strip().upper() != "BARRANCABERMEJA":
                    raise AssertionError("Novedades 2V: municipio discordante")
                fila.setdefault("potencial", "")
                fila.setdefault("sufragantesSegunE11", "")
                fila["mesa"] = fila["mesaNumero"].zfill(6)
                fila["cod_zona"] = fila["zonaNombre"].zfill(2)
                fila["cod_puesto"] = fila["puestoCodigo"].zfill(2)
                novedades_rows.append(fila)
    nov_2v = pd.DataFrame(novedades_rows)
    if nov_2v.empty or nov_2v.duplicated(MESAS).any():
        raise AssertionError("Novedades 2V vacías o con mesas duplicadas")
    if not nov_2v.totalVotos.str.fullmatch(r"\d+").all():
        raise ValueError("Novedades 2V: totalVotos no entero")
    control_2v = (esc_2v.groupby(MESAS, as_index=False).votos.sum()
                  .rename(columns={"votos": "votos_escrutinio"}))
    pcontrol = (pre_2v.groupby(MESAS, as_index=False).votos.sum()
                .rename(columns={"votos": "votos_preconteo"}))
    control_2v = control_2v.merge(pcontrol, on=MESAS, how="outer", validate="one_to_one")
    control_2v = control_2v.merge(nov_2v[MESAS + ["totalVotos", "potencial", "sufragantesSegunE11"]],
                                  on=MESAS, how="outer", validate="one_to_one")
    if control_2v[["votos_escrutinio", "votos_preconteo", "totalVotos"]].isna().any().any():
        raise AssertionError("Mesas ausentes en uno de los tres componentes de 2V")
    control_2v["totalVotos"] = control_2v.totalVotos.astype(int)
    control_2v["diferencia_escrutinio_preconteo"] = (
        control_2v.votos_escrutinio - control_2v.votos_preconteo)
    control_2v["cuadra_con_novedades"] = control_2v.votos_escrutinio.eq(control_2v.totalVotos)
    if not control_2v.cuadra_con_novedades.all():
        raise AssertionError("Escrutinio 2V no concilia con novedades en todas las mesas")
    if len(control_2v) != int(local_2v.mesas_declaradas.astype(int).sum()):
        raise AssertionError("Número de mesas 2V distinto de DIVIPOL")
    por_puesto = control_2v.groupby(["cod_zona", "cod_puesto"]).size().to_dict()
    if any(por_puesto.get((r.cod_zona, r.cod_puesto), 0) != int(r.mesas_declaradas)
           for r in local_2v.itertuples()):
        raise AssertionError("Mesas por puesto 2V distintas de DIVIPOL")

    salida = RAIZ / "datos" / "intermedios" / "registraduria" / "2026"
    salida.mkdir(parents=True, exist_ok=True)
    salidas = {"preconteo": pre_2v, "escrutinio": esc_2v, "novedades": nov_2v,
               "control_mesas": control_2v, "divipol_local": local_2v}
    rutas = {}
    for nombre, datos in salidas.items():
        ruta_csv = salida / f"presidencia_2026_2v_{nombre}_barrancabermeja.csv"
        temporal = ruta_csv.with_suffix(".csv.part")
        try:
            datos.to_csv(temporal, index=False, encoding="utf-8-sig")
            temporal.replace(ruta_csv)
        finally:
            temporal.unlink(missing_ok=True)
        rutas[nombre] = ruta_csv.relative_to(RAIZ).as_posix()
    resultado = {"id_fuente": "REG_2026_PRES_2V", "estado": "conciliada",
                 "zip_sha256": adquisicion["sha256"], "pdf_estructura_sha256": huella_documento,
                 "sha256_oficial_votos": huellas_2v, "sha256_oficial_novedades": huella_nov,
                 "archivos_basicos_con_hash_verificado": len(entradas),
                 "puestos_divipol": len(local_2v), "mesas_divipol": len(control_2v),
                 "preconteo": resumen_pre_2v, "escrutinio": resumen_esc_2v,
                 "mesas_con_diferencia_preconteo_escrutinio": int(
                     control_2v.diferencia_escrutinio_preconteo.ne(0).sum()),
                 "diferencia_neta_votos": int(control_2v.diferencia_escrutinio_preconteo.sum()),
                 "salidas": rutas}
    (AUDITORIA / "control_calidad_2026_2v.json").write_text(
        json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")
    return resultado

qa_2v = {"estado": "sin_original"}
adq_2v = next(d for d in adquisiciones if d["id_fuente"] == "REG_2026_PRES_2V")
if adq_2v["estado"] in {"existente_validado", "descargado_validado", "manual_validado"}:
    if not PILOTO_DISPONIBLE:
        qa_2v = {"estado": "pendiente_piloto", "detalle": "Se necesita el PDF auditado de 1V"}
    else:
        try:
            qa_2v = procesar_presidencia_2026_2v(adq_2v)
        except (OSError, ValueError, AssertionError, KeyError, IndexError,
                UnicodeError, zipfile.BadZipFile) as exc:
            qa_2v = {"estado": "pendiente_interpretacion",
                     "detalle": f"{type(exc).__name__}: {exc}"}
print("Presidencia 2026 2V:", qa_2v["estado"], qa_2v.get("detalle", ""))
if qa_2v["estado"] == "conciliada":
    print("Mesas:", qa_2v["mesas_divipol"], "| Votos escrutinio:",
          qa_2v["escrutinio"]["votos_locales"])

Presidencia 2026 2V: conciliada 
Mesas: 566 | Votos escrutinio: 130340


### 6.2 Congreso 2026, preconteo y escrutinio

El paquete oficial contiene cuatro archivos de preconteo y un ZIP con los archivos de escrutinio. El adaptador verifica las huellas SHA-256, identifica `27/019` en DIVIPOL y conserva separados los códigos de corporación y circunscripción. Registra la extracción cuando encuentra datos de Cámara y Senado en ambos tipos de resultado. La conciliación por corporación se realiza durante el perfilado.

In [11]:
def huellas_sha256_manifiesto(contenido: bytes) -> set[str]:
    """Lee las huellas SHA-256 independientemente de la codificación del HASH.

    El ZIP y cada archivo siguen validándose por separado; la asociación se
    hace por la huella del contenido y evita depender del formato de la etiqueta
    Filename, que cambia entre elecciones.
    """
    if contenido.startswith((b"\xff\xfe", b"\xfe\xff")) or contenido[:2048].count(b"\x00") > 256:
        texto = contenido.decode("utf-16")
    else:
        try:
            texto = contenido.decode("utf-8-sig")
        except UnicodeDecodeError:
            texto = contenido.decode("cp1252")
    return {h.lower() for h in re.findall(
        r"(?<![0-9a-fA-F])[0-9a-fA-F]{64}(?![0-9a-fA-F])", texto)}


def extraer_congreso_2026(adquisicion: dict) -> dict:
    """Lee MMV por formato publicado; preserva códigos corporación/circunscripción."""
    ruta = RAIZ / adquisicion["ruta_relativa"]
    with zipfile.ZipFile(ruta) as z:
        pdf = miembro_unico(z, "Estructuras Basicas (1808).pdf")
        if sha256_bytes(z.read(pdf)) != huella_documento:
            raise ValueError("El PDF de Congreso difiere del esquema MMV 2026 auditado")
        basicos_nombres = [n for n in z.namelist() if re.search(r"(?i)archivosbasicos.*\.zip$", n)]
        inner_names = [n for n in z.namelist() if re.search(r"(?i)mmvescrutinio.*\.zip$", n)]
        pre = [n for n in z.namelist() if re.search(r"(?i)/(CAM|CNS|CTP|SEN)_MMV_9999\.txt$", n)]
        hashpre = [n for n in z.namelist() if re.search(r"(?i)/mmvpreconteocongreso2026/HASH\.txt$", n)]
        if (len(basicos_nombres), len(inner_names), len(pre), len(hashpre)) != (1, 1, 4, 1):
            raise ValueError("Archivos MMV/Básicos/HASH de Congreso incompletos o ambiguos")
        with zipfile.ZipFile(io.BytesIO(z.read(basicos_nombres[0]))) as b:
            huellas_basicos = huellas_sha256_manifiesto(b.read(miembro_unico(b, "HASH.txt")))
            archivos_txt = [n for n in b.namelist() if n.upper().endswith(".TXT") and
                            PurePosixPath(n).name.upper() != "HASH.TXT"]
            if len(archivos_txt) != 6 or not huellas_basicos:
                raise AssertionError(
                    f"Básicos de Congreso: {len(archivos_txt)} TXT, "
                    f"{len(huellas_basicos)} huellas SHA-256 reconocibles en HASH.txt")
            for nombre in archivos_txt:
                with b.open(nombre) as stream:
                    calculada = hashlib.file_digest(stream, "sha256").hexdigest()
                if calculada not in huellas_basicos:
                    raise AssertionError("El HASH de básicos no cubre " + PurePosixPath(nombre).name)
            div = leer_fijo(b.read(miembro_por_prefijo(b, "DIVIPOL_", ".TXT")),
                            DIVIPOL_CAMPOS, "DIVIPOL_CONGRESO_2026")
        local_div = div.loc[div.departamento.str.upper().eq("SANTANDER") &
                            div.municipio.str.upper().eq("BARRANCABERMEJA")].copy()
        codigos = local_div[["cod_depto", "cod_municipio"]].drop_duplicates()
        if len(codigos) != 1 or tuple(codigos.iloc[0]) != (COD_DEPTO, COD_MUNICIPIO):
            raise AssertionError("DIVIPOL Congreso no identifica unívocamente Barrancabermeja")
        if local_div.duplicated(geocampos).any():
            raise AssertionError("Puestos repetidos en DIVIPOL Congreso")
        puestos_c = {tuple(r[c] for c in geocampos): r for r in local_div.to_dict("records")}

        huellas_pre = huellas_sha256_manifiesto(z.read(hashpre[0]))
        if not huellas_pre:
            raise AssertionError("HASH de preconteo sin huellas SHA-256 legibles")
        registros_pre, resumen_pre = [], {}
        for nombre in pre:
            grupo = PurePosixPath(nombre).name.split("_", 1)[0].upper()
            with z.open(nombre) as stream:
                calculada = hashlib.file_digest(stream, "sha256").hexdigest()
            if calculada not in huellas_pre:
                raise AssertionError("SHA-256 de preconteo discordante: " + nombre)
            nacionales, locales = 0, 0
            with z.open(nombre) as stream:
                for nacionales, linea in enumerate(stream, start=1):
                    linea = linea.rstrip(b"\r\n")
                    if len(linea) != 38:
                        raise ValueError(f"{grupo}, línea {nacionales}: ancho MMV distinto de 38")
                    if linea[:5] != (COD_DEPTO + COD_MUNICIPIO).encode("ascii"):
                        continue
                    fila = interpretar_registro(linea, "preconteo", nacionales)
                    geo = tuple(fila[c] for c in geocampos)
                    if geo not in puestos_c:
                        raise AssertionError(f"Puesto preconteo ausente de DIVIPOL: {geo}")
                    registros_pre.append({"_id_fuente": "REG_2026_CONGRESO", "_tipo_resultado": "preconteo",
                                          "_grupo_archivo": grupo, "_miembro_origen": nombre, **fila,
                                          "cod_corporacion": "por_enlazar_con_catalogo",
                                          "municipio": "BARRANCABERMEJA", "departamento": "SANTANDER"})
                    locales += 1
            resumen_pre[grupo] = {"filas_nacionales": nacionales, "filas_locales": locales,
                                  "sha256_oficial": calculada}

        nested_info = z.getinfo(inner_names[0])
        if shutil.disk_usage(AUDITORIA).free < nested_info.file_size + 1024**3:
            raise OSError("Falta espacio para ZIP anidado y reserva de 1 GiB")
        temporal_inner = None
        try:
            with tempfile.NamedTemporaryFile(dir=AUDITORIA, suffix=".zip.part", delete=False) as tmp:
                temporal_inner = Path(tmp.name)
                with z.open(inner_names[0]) as stream:
                    shutil.copyfileobj(stream, tmp, length=1024 * 1024)
        except BaseException:
            if temporal_inner is not None:
                temporal_inner.unlink(missing_ok=True)
            raise

    registros_esc, resumen_esc = [], {}
    try:
        with zipfile.ZipFile(temporal_inner) as interior:
            archivos = [m for m in interior.infolist() if not m.is_dir() and
                        m.filename.lower().endswith((".csv", ".txt")) and
                        "hash" not in PurePosixPath(m.filename).name.lower()]
            if not archivos:
                raise ValueError("El ZIP de escrutinio no contiene miembros tabulares")
            for miembro in archivos:
                nacionales, locales = 0, 0
                with interior.open(miembro) as stream:
                    for nacionales, linea in enumerate(stream, start=1):
                        if not linea.startswith((b"9999;" + COD_DEPTO.encode() + b";" + COD_MUNICIPIO.encode() + b";")):
                            continue
                        campos = linea.decode("utf-8").strip().split(";")
                        if len(campos) != 13 or campos[-1] != "":
                            raise ValueError(f"Escrutinio, línea {nacionales}: no tiene 12 campos y ; final")
                        fila = dict(zip(ESC_CAMPOS, campos[:-1]))
                        if not (fila["cod_corporacion"].isdigit() and fila["votos"].isdigit()):
                            raise ValueError("Corporación o votos de escrutinio no numéricos")
                        fila["cod_zona"] = normalizar_zona_escrutinio(fila["cod_zona"])
                        fila["cod_partido"] = fila["cod_partido"].zfill(5)
                        geo = tuple(fila[c] for c in geocampos)
                        if geo not in puestos_c:
                            raise AssertionError(f"Puesto escrutinio ausente de DIVIPOL: {geo}")
                        registros_esc.append({"_id_fuente": "REG_2026_CONGRESO",
                                              "_tipo_resultado": "escrutinio", "_miembro_origen": miembro.filename,
                                              **fila, "municipio": "BARRANCABERMEJA",
                                              "departamento": "SANTANDER"})
                        locales += 1
                resumen_esc[miembro.filename] = {"filas_nacionales": nacionales,
                                                  "filas_locales": locales}
    finally:
        temporal_inner.unlink(missing_ok=True)

    pre_tabla, esc_tabla = pd.DataFrame(registros_pre), pd.DataFrame(registros_esc)
    if pre_tabla.empty or esc_tabla.empty:
        raise AssertionError("Preconteo o escrutinio local de Congreso vacío")
    if not {"CAM", "SEN"}.issubset(set(pre_tabla._grupo_archivo)):
        raise AssertionError("Sin cobertura CAM y SEN en preconteo local")
    llave_pre = ["_grupo_archivo", "cod_depto", "cod_municipio", "cod_zona", "cod_puesto",
                 "mesa", "cod_circunscripcion", "cod_partido", "cod_candidato"]
    llave_esc = ["cod_corporacion", "cod_circunscripcion", "cod_depto", "cod_municipio",
                 "cod_zona", "cod_puesto", "mesa", "cod_partido", "cod_candidato"]
    if pre_tabla.duplicated(llave_pre).any() or esc_tabla.duplicated(llave_esc).any():
        raise AssertionError("Llaves MMV duplicadas en Congreso 2026")
    destino = RAIZ / "datos" / "intermedios" / "congreso" / "2026"
    destino.mkdir(parents=True, exist_ok=True)
    salidas = {}
    for nombre, tabla in {"preconteo": pre_tabla, "escrutinio": esc_tabla,
                          "divipol_local": local_div}.items():
        ruta_csv = destino / f"congreso_2026_{nombre}_barrancabermeja.csv"
        temporal = ruta_csv.with_suffix(".csv.part")
        try:
            tabla.to_csv(temporal, index=False, encoding="utf-8-sig")
            temporal.replace(ruta_csv)
        finally:
            temporal.unlink(missing_ok=True)
        salidas[nombre] = ruta_csv.relative_to(RAIZ).as_posix()
    qa = {"id_fuente": "REG_2026_CONGRESO", "estado": "extraida_pendiente_perfilado",
          "zip_sha256": adquisicion["sha256"], "pdf_estructura_sha256": huella_documento,
          "archivos_basicos_verificados": len(archivos_txt), "puestos_divipol": len(local_div),
          "filas_preconteo_locales": len(pre_tabla), "filas_escrutinio_locales": len(esc_tabla),
          "grupos_preconteo": resumen_pre, "archivos_escrutinio": resumen_esc,
          "votos_preconteo": int(pre_tabla.votos.astype("int64").sum()),
          "votos_escrutinio": int(esc_tabla.votos.astype("int64").sum()),
          "limite": "Corporaciones y circunscripciones se perfilan en 01; no se concilian como una sola elección",
          "salidas": salidas}
    (AUDITORIA / "control_congreso_2026.json").write_text(
        json.dumps(qa, ensure_ascii=False, indent=2), encoding="utf-8")
    return qa

qa_congreso_2026 = {"estado": "sin_original"}
adq_congreso_2026 = next(d for d in adquisiciones if d["id_fuente"] == "REG_2026_CONGRESO")
if adq_congreso_2026["estado"] in {"existente_validado", "descargado_validado", "manual_validado"}:
    if not PILOTO_DISPONIBLE:
        qa_congreso_2026 = {"estado": "pendiente_piloto",
                             "detalle": "Se necesita el PDF auditado de Presidencia 2026 1V"}
    else:
        try:
            qa_congreso_2026 = extraer_congreso_2026(adq_congreso_2026)
        except (OSError, ValueError, AssertionError, KeyError, IndexError,
                UnicodeError, zipfile.BadZipFile) as exc:
            qa_congreso_2026 = {"estado": "pendiente_interpretacion",
                                 "detalle": f"{type(exc).__name__}: {exc}"}
print("Congreso 2026:", qa_congreso_2026["estado"], qa_congreso_2026.get("detalle", ""))

Congreso 2026: extraida_pendiente_perfilado 


## 7. Extracción de tablas con cabecera

En otros CSV, el cuaderno identifica la cabecera y las columnas de municipio y votos o potencial. Exige `BARRANCABERMEJA` y, si el archivo incluye varios departamentos, `SANTANDER`. Si no hay columna de departamento, acepta el archivo solo cuando su nombre o URL confirma que corresponde a Santander. Conserva como texto los códigos y sus ceros iniciales. Cada archivo genera una salida independiente. El tipo de resultado queda como `por_verificar` cuando no se indica en el nombre. Las filas extraídas se guardan como datos intermedios para su revisión posterior.

Las hojas XLSX del DANE se procesan por lotes cuando sus encabezados y el municipio se identifican sin ambigüedad. Se conserva el número de fila y el nombre de la hoja de origen. Los archivos de ancho fijo y los esquemas desconocidos quedan como `pendiente_interpretacion` y no se cuentan como tablas procesadas. Se incorporan cuando se conoce su estructura.

In [12]:
import unicodedata

def canon(nombre: str) -> str:
    texto = unicodedata.normalize("NFKD", str(nombre))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"[^A-Z0-9]+", "_", texto.upper()).strip("_")

def columna_nominal(columnas: list[str], patrones: tuple[str, ...]) -> str | None:
    for col in columnas:
        nombre = canon(col)
        if any(p in nombre for p in patrones) and not any(
            c in nombre for c in ("CODIGO", "COD_", "_COD", "ID_", "_ID")
        ):
            return col
    return None

def esquema_cabecera(muestra: bytes, familia: str) -> dict:
    if muestra.startswith((b"\xff\xfe", b"\xfe\xff")):
        codec = "utf-16"
    elif muestra.startswith(b"\xef\xbb\xbf"):
        codec = "utf-8-sig"
    else:
        try:
            muestra.decode("utf-8")
            codec = "utf-8"
        except UnicodeDecodeError:
            codec = "cp1252"
    texto = muestra.decode(codec, errors="replace")
    candidatas = []
    for n, linea in enumerate(texto.splitlines()[:200]):
        for sep in (",", ";", "\t", "|"):
            if linea.count(sep) < 3:
                continue
            try:
                columnas = next(csv.reader([linea], delimiter=sep))
            except csv.Error:
                continue
            if len(columnas) < 4 or len(set(columnas)) != len(columnas):
                continue
            municipio = columna_nominal(columnas, ("MUNICIPIO", "MPIO"))
            depto = columna_nominal(columnas, ("DEPARTAMENTO", "DEPTO"))
            votos = next((col for col in columnas if
                ("VOTO" in canon(col) or "VOTACION" in canon(col)) and
                not any(contexto in canon(col) for contexto in
                        ("PUESTO", "MESA", "POTENCIAL", "LUGAR", "ZONA"))), None)
            potencial = columna_nominal(columnas, ("POTENCIAL", "CENSO_ELECTORAL"))
            mesa = columna_nominal(columnas, ("MESA",))
            puesto = columna_nominal(columnas, ("PUESTO",))
            candidato = columna_nominal(columnas, ("CANDIDATO", "AGRUPACION", "PARTIDO"))
            cargo = columna_nominal(columnas, ("CORPORACION", "CARGO", "DIGNIDAD"))
            persona = next((c for c in columnas if ("CANDIDATO" in canon(c) or "NOMBRE" in canon(c))
                            and not any(k in canon(c) for k in ("MUNICIPIO", "DEPARTAMENTO", "PARTIDO"))), None)
            relevante = (bool(votos) and bool(mesa or candidato)) if familia in {
                "territorial", "presidencia", "congreso"
            } else (bool(potencial) and bool(puesto)) if familia == "censo_electoral" else (
                bool(cargo) and bool(persona)) if familia == "candidaturas" else False
            if municipio and relevante:
                candidatas.append({"codificacion": codec, "separador": sep,
                                   "linea_cabecera": n, "columnas": columnas,
                                   "col_municipio": municipio, "col_departamento": depto,
                                   "col_votos": votos, "col_potencial": potencial,
                                   "col_cargo": cargo, "col_persona": persona,
                                   "tiene_mesa": bool(mesa), "tiene_candidato": bool(candidato),
                                   "puntaje": 3 * bool(depto) + 2 * bool(mesa) + 2 * bool(candidato)})
    return max(candidatas, key=lambda d: d["puntaje"]) if candidatas else {
        "codificacion": codec, "motivo": "sin cabecera nominal y columnas suficientes"}

def token_normalizado(serie: pd.Series) -> pd.Series:
    return serie.astype(str).map(canon)

def nombre_seguro(nombre: str) -> str:
    simple = unicodedata.normalize("NFKD", nombre)
    simple = "".join(x for x in simple if not unicodedata.combining(x))
    return re.sub(r"[^a-z0-9]+", "_", simple.lower()).strip("_")[:85] or "datos"

def seleccionar_cobertura(fuente: dict, tablas: list[dict], diagnosticos: list[dict]) -> tuple[str, str]:
    if fuente["id_fuente"] == "REG_CENSO_HIST":
        requeridos = {"2019", "2022", "2023", "2026"}
        hallados = set().union(*(set(t.get("anios_identificados", [])) for t in tablas))
        if requeridos.issubset(hallados):
            return "anios_identificados", ", ".join(sorted(hallados))
        return "anios_no_verificados", ("Faltan años del censo: " +
            ", ".join(sorted(requeridos - hallados)) + "; observados: " +
            (", ".join(sorted(hallados)) or "ninguno"))
    # Las corporaciones esperadas se consideran cobertura independiente.
    if fuente["id_fuente"] not in {"REG_2023_TERR", "REG_2019_TERR", "REG_2022_CONGRESO",
                                   "REG_2026_CONGRESO", "REG_CANDIDATURAS_2019", "REG_CANDIDATURAS_2023"}:
        return "aplica_una_eleccion", ""
    objetivo = {"ALCALDIA", "CONCEJO"} if fuente["familia"] in {"territorial", "candidaturas"} else {"SENADO", "CAMARA"}
    evidencia = set()
    for fila in tablas:
        evidencia.update(fila.get("corporaciones_identificadas", []))
    if objetivo.issubset(evidencia):
        return "corporaciones_identificadas", ", ".join(sorted(evidencia))
    return "corporaciones_no_verificadas", "Se requieren " + ", ".join(sorted(objetivo)) + "; identificadas: " + ", ".join(sorted(evidencia))

def extraer_xlsx(fuente: dict, ruta: Path) -> tuple[list[dict], list[dict]]:
    try:
        from openpyxl import load_workbook
    except ImportError:
        return [], [{"id_fuente": fuente["id_fuente"], "miembro": ruta.name,
                     "estado": "falta_openpyxl", "detalle": "Instala openpyxl en el kernel del proyecto"}]
    resultado, diagn = [], []
    libro = load_workbook(ruta, read_only=True, data_only=True)
    try:
        for hoja in libro:
            registro = {"id_fuente": fuente["id_fuente"], "miembro": hoja.title,
                        "estado": "", "detalle": "", "filas_leidas": 0, "filas_locales": 0}
            iterador = hoja.iter_rows(values_only=True)
            cabecera, numero_cabecera = None, 0
            for n, row in enumerate(iterador, start=1):
                if n > 100:
                    break
                cols = [str(x).strip() if x is not None else "" for x in row]
                nombres = [canon(x) for x in cols]
                if len(set(c for c in nombres if c)) < 3:
                    continue
                if any("MUNICIPIO" in c or c in {"MPIO", "DPMP", "CODMPIO"} for c in nombres):
                    cabecera, numero_cabecera = cols, n
                    break
            if cabecera is None:
                registro.update(estado="esquema_no_resuelto", detalle="No hay encabezado municipal en las primeras 100 filas")
                diagn.append(registro)
                continue
            normal = [canon(c) for c in cabecera]
            if len(set(normal)) != len(normal) or not all(normal):
                registro.update(estado="esquema_no_resuelto", detalle="Encabezados vacíos o duplicados")
                diagn.append(registro)
                continue
            col_nombre = next((i for i, c in enumerate(normal) if "MUNICIPIO" in c and "COD" not in c), None)
            col_dane = next((i for i, c in enumerate(normal) if c in {"DPMP", "CODIGO_DANE", "CODIGO_MUNICIPIO", "COD_MUNICIPIO", "CODMPIO"}), None)
            col_depto = next((i for i, c in enumerate(normal) if "DEPARTAMENTO" in c and "COD" not in c), None)
            if col_nombre is None and col_dane is None:
                registro.update(estado="esquema_no_resuelto", detalle="No se identifica nombre ni código de municipio")
                diagn.append(registro)
                continue
            salida = (RAIZ / "datos" / "intermedios" / fuente["familia"] /
                      (str(fuente["anio"]) if fuente["anio"] else "serie") / fuente["id_fuente"] /
                      (nombre_seguro(hoja.title) + "_barrancabermeja.csv"))
            salida.parent.mkdir(parents=True, exist_ok=True)
            temporal = salida.with_suffix(".csv.part")
            temporal.unlink(missing_ok=True)
            try:
                with temporal.open("w", newline="", encoding="utf-8-sig") as archivo:
                    escritor = csv.writer(archivo)
                    escritor.writerow(["_id_fuente", "_hoja_origen", "_fila_excel", *cabecera])
                    for numero, row in enumerate(iterador, start=numero_cabecera + 1):
                        registro["filas_leidas"] += 1
                        valores = ["" if x is None else str(x) for x in row[:len(cabecera)]]
                        valores += [""] * (len(cabecera) - len(valores))
                        nombre = canon(valores[col_nombre]) if col_nombre is not None else ""
                        codigo = re.sub(r"\.0$", "", valores[col_dane].strip()) if col_dane is not None else ""
                        if col_nombre is not None and nombre != "BARRANCABERMEJA":
                            continue
                        if col_depto is not None and canon(valores[col_depto]) != "SANTANDER":
                            continue
                        if col_nombre is None and codigo != "68081":
                            continue
                        if len(codigo) == 5 and codigo != "68081":
                            raise ValueError(f"Nombre municipal y código DANE discordantes: fila {numero}")
                        escritor.writerow([fuente["id_fuente"], hoja.title, numero, *valores])
                        registro["filas_locales"] += 1
                if registro["filas_locales"]:
                    temporal.replace(salida)
                    registro.update(estado="extraido_pendiente_perfilado",
                                    ruta_relativa=salida.relative_to(RAIZ).as_posix(),
                                    sha256=sha256_archivo(salida), columnas=cabecera,
                                    detalle="Filtrado municipal en XLSX; verificar series, años y unidad en 01")
                    resultado.append(registro.copy())
                else:
                    registro.update(estado="sin_filas_locales", detalle="No hay filas del municipio en esta hoja")
            except (OSError, ValueError) as exc:
                registro.update(estado="error_extraccion", detalle=f"{type(exc).__name__}: {exc}")
            finally:
                temporal.unlink(missing_ok=True)
            diagn.append(registro)
    finally:
        libro.close()
    return resultado, diagn

def proceso_generico(fuente: dict, adquisicion: dict) -> tuple[list[dict], list[dict]]:
    resultado, diagn = [], []
    if adquisicion["estado"] not in {"existente_validado", "descargado_validado", "manual_validado"}:
        return resultado, diagn
    if fuente["id_fuente"] in {"REG_2026_PRES_1V", "REG_2026_PRES_2V", "REG_2026_CONGRESO",
                               "REG_2022_PRES_1V", "REG_2022_PRES_2V",
                               "DANE_MUN_EDAD_2018_2042", "DANE_MUN_EDAD_2005_2017"}:
        return resultado, diagn  # Adaptador específico de la sección 6.
    ruta = RAIZ / adquisicion["ruta_relativa"]
    if fuente["extension"] == "xlsx":
        return extraer_xlsx(fuente, ruta)
    if fuente["extension"] == "zip":
        z = zipfile.ZipFile(ruta)
        miembros = [m.filename for m in z.infolist()
                    if not m.is_dir() and m.filename.lower().endswith((".csv", ".txt", ".tsv"))]
        def abrir(miembro):
            return z.open(miembro)
    else:
        z = None
        miembros = [ruta.name]
        def abrir(miembro):
            return ruta.open("rb")
    try:
        for miembro in miembros:
            registro = {"id_fuente": fuente["id_fuente"], "miembro": miembro,
                        "estado": "", "detalle": "", "filas_leidas": 0, "filas_locales": 0}
            with abrir(miembro) as stream:
                muestra = stream.read(256 * 1024)
            esquema = esquema_cabecera(muestra, fuente["familia"])
            registro["codificacion"] = esquema.get("codificacion", "")
            registro["columnas"] = esquema.get("columnas", [])
            if not esquema.get("col_municipio"):
                registro.update(estado="esquema_no_resuelto", detalle=esquema.get("motivo", ""))
                diagn.append(registro)
                continue
            # Solo una fuente cuyo recurso publicado indica Santander permite carecer de depto nominal.
            recurso_depto = "SANTANDER" in (miembro + adquisicion["url_origen"]).upper()
            if not esquema["col_departamento"] and not recurso_depto:
                registro.update(estado="departamento_no_resuelto",
                                detalle="Falta departamento nominal y el recurso no identifica Santander")
                diagn.append(registro)
                continue
            registro.update(codificacion=esquema["codificacion"], separador=esquema["separador"],
                            linea_cabecera=esquema["linea_cabecera"],
                            col_municipio=esquema["col_municipio"],
                            col_departamento=esquema["col_departamento"] or "")
            corporaciones = set()
            anios_censo = set()
            nombre_miembro = canon(miembro)
            for corporacion in ("ALCALDIA", "CONCEJO", "SENADO", "CAMARA"):
                if corporacion in nombre_miembro:
                    corporaciones.add(corporacion)
            salida = (RAIZ / "datos" / "intermedios" / fuente["familia"] /
                      (str(fuente["anio"]) if fuente["anio"] else "serie") / fuente["id_fuente"] /
                      (nombre_seguro(PurePosixPath(miembro).name) + "_barrancabermeja.csv"))
            salida.parent.mkdir(parents=True, exist_ok=True)
            temporal = salida.with_suffix(".csv.part")
            temporal.unlink(missing_ok=True)
            try:
                with abrir(miembro) as stream:
                    iterador = pd.read_csv(stream, sep=esquema["separador"],
                                           skiprows=esquema["linea_cabecera"],
                                           encoding=esquema["codificacion"], dtype=str,
                                           keep_default_na=False, chunksize=100_000,
                                           on_bad_lines="error")
                    for bloque in iterador:
                        registro["filas_leidas"] += len(bloque)
                        esperadas = {esquema["col_municipio"]}
                        if esquema["col_departamento"]:
                            esperadas.add(esquema["col_departamento"])
                        if not esperadas.issubset(bloque.columns):
                            raise ValueError("Las columnas detectadas no aparecen al leer la tabla completa")
                        mascara = token_normalizado(bloque[esquema["col_municipio"]]).eq("BARRANCABERMEJA")
                        if esquema["col_departamento"]:
                            mascara &= token_normalizado(bloque[esquema["col_departamento"]]).eq("SANTANDER")
                        local = bloque.loc[mascara].copy()
                        if local.empty:
                            continue
                        if fuente["id_fuente"] == "REG_CENSO_HIST":
                            col_anio = columna_nominal(list(local.columns), ("ANO", "VIGENCIA"))
                            if col_anio:
                                anios_censo.update(v.strip() for v in local[col_anio].unique()
                                                   if re.fullmatch(r"20\d{2}", str(v).strip()))
                        col_corporacion = columna_nominal(list(local.columns), ("CORPORACION",))
                        if col_corporacion:
                            for valor in local[col_corporacion].drop_duplicates():
                                normal = canon(valor)
                                for corporacion in ("ALCALDIA", "CONCEJO", "SENADO", "CAMARA"):
                                    if corporacion in normal:
                                        corporaciones.add(corporacion)
                        col_valor = (esquema["col_potencial"] if fuente["familia"] == "censo_electoral"
                                     else esquema["col_votos"] or esquema["col_potencial"])
                        if fuente["familia"] == "candidaturas":
                            col_cargo = esquema["col_cargo"]
                            cargo_normalizado = token_normalizado(local[col_cargo])
                            local = local.loc[cargo_normalizado.str.contains(r"ALCALDIA|CONCEJO", regex=True)].copy()
                            if local.empty:
                                continue
                            for valor in local[col_cargo].drop_duplicates():
                                for corporacion in ("ALCALDIA", "CONCEJO"):
                                    if corporacion in canon(valor):
                                        corporaciones.add(corporacion)
                            if local[esquema["col_persona"]].astype(str).str.strip().eq("").any():
                                raise ValueError("Candidatura sin nombre en filas locales")
                        elif not local[col_valor].astype(str).str.fullmatch(r"\d+").all():
                            raise ValueError(f"Valor no entero en {col_valor}")
                        local.insert(0, "_miembro_origen", miembro)
                        local.insert(0, "_tipo_resultado", (
                            "preconteo" if "PRECONTEO" in miembro.upper() else
                            "escrutinio" if "ESCRUTINIO" in miembro.upper() else "por_verificar"))
                        local.insert(0, "_id_fuente", fuente["id_fuente"])
                        primera = not temporal.exists()
                        local.to_csv(temporal, mode="a", index=False, header=primera,
                                     encoding="utf-8-sig" if primera else "utf-8")
                        registro["filas_locales"] += len(local)
                if registro["filas_locales"]:
                    temporal.replace(salida)
                    registro.update(estado="extraido_pendiente_perfilado", ruta_relativa=salida.relative_to(RAIZ).as_posix(),
                                    sha256=sha256_archivo(salida),
                                    corporaciones_identificadas=sorted(corporaciones),
                                    anios_identificados=sorted(anios_censo),
                                    detalle="Filtrado nominal; aún faltan contratos de esquema y conciliación por elección")
                    resultado.append(registro.copy())
                else:
                    registro.update(estado="sin_filas_locales", detalle="Tabla legible sin Barrancabermeja; revisar cobertura")
            except (OSError, UnicodeError, ValueError, pd.errors.ParserError,
                    pd.errors.EmptyDataError) as exc:
                registro.update(estado="error_extraccion", detalle=f"{type(exc).__name__}: {exc}")
            finally:
                temporal.unlink(missing_ok=True)
            diagn.append(registro)
    finally:
        if z is not None:
            z.close()
    if not miembros:
        diagn.append({"id_fuente": fuente["id_fuente"], "miembro": "",
                      "estado": "sin_tabulares_csv", "detalle": "ZIP sin CSV/TXT/TSV identificables"})
    return resultado, diagn

tablas_genericas, diagnosticos_genericos = [], []
for fuente, adquisicion in zip(FUENTES, adquisiciones):
    try:
        tablas, diagnosticos = proceso_generico(fuente, adquisicion)
    except (OSError, ValueError, zipfile.BadZipFile) as exc:
        tablas, diagnosticos = [], [{"id_fuente": fuente["id_fuente"],
                                    "estado": "error_inventario_tabular", "detalle": str(exc)}]
    tablas_genericas.extend(tablas)
    diagnosticos_genericos.extend(diagnosticos)
tabla_esquemas = pd.DataFrame(diagnosticos_genericos)
tabla_locales = pd.DataFrame(tablas_genericas)
print("Tablas nuevas filtradas:", len(tablas_genericas), "| archivos con diagnóstico:", len(diagnosticos_genericos))
display(tabla_esquemas[[c for c in ["id_fuente", "miembro", "estado", "filas_leidas",
                                      "filas_locales", "detalle"] if c in tabla_esquemas.columns]].head(30))

Tablas nuevas filtradas: 0 | archivos con diagnóstico: 2


,id_fuente,miembro,estado,detalle
0,REG_2018_PRES_1V,,sin_tabulares_csv,ZIP sin CSV/TXT/TSV identificables
1,REG_2018_PRES_2V,,sin_tabulares_csv,ZIP sin CSV/TXT/TSV identificables


### 7.1 Presidencia 2022 y proyecciones del DANE

Cada ronda presidencial de 2022 contiene un CSV de 16 columnas separado por punto y coma. Se extraen las filas de Barrancabermeja, se verifican los códigos electorales y se deja el tipo de resultado como `por_verificar` hasta documentarlo. Para las proyecciones del DANE, se conservan las columnas de edad y sexo y se comprueba el código municipal `68081`, la presencia de las tres áreas y la consistencia de sus sumas. La población proyectada no se usa como denominador electoral.

In [13]:
def extraer_presidencia_2022(fuente: dict, adquisicion: dict) -> dict:
    """Extrae el CSV histórico con los 16 campos observados en la fuente oficial."""
    ruta = RAIZ / adquisicion["ruta_relativa"]
    with zipfile.ZipFile(ruta) as z:
        miembros = [x for x in z.namelist() if x.lower().endswith(".csv")]
        if len(miembros) != 1:
            raise ValueError(f"Se esperaba un CSV, se encontraron {miembros}")
        miembro = miembros[0]
        with z.open(miembro) as stream:
            reader = csv.DictReader(io.TextIOWrapper(stream, encoding="cp1252", newline=""),
                                    delimiter=";")
            columnas = [x.lstrip("\ufeff").strip() for x in (reader.fieldnames or [])]
            esperadas = ["DEP", "DEPNOMBRE", "MUN", "MUNNOMBRE", "ZONA", "PUESTO",
                         "PUESNOMBRE", "MESA", "CORCODIGO", "CORNOMBRE", "CIR",
                         "PAR", "PARNOMBRE", "CAN", "CANNOMBRE", "VOTOS"]
            if columnas != esperadas:
                raise ValueError(f"Columnas históricas inesperadas: {columnas}")
            registros, nacionales = [], 0
            for nacionales, fila in enumerate(reader, start=1):
                if None in fila or any(v is None for v in fila.values()):
                    raise ValueError(f"Registro CSV mal formado en línea {nacionales + 1}")
                if fila["MUNNOMBRE"].strip().upper() != "BARRANCABERMEJA":
                    continue
                if fila["DEPNOMBRE"].strip().upper() != "SANTANDER":
                    raise AssertionError("Barrancabermeja asociada con otro departamento")
                if fila["CORCODIGO"].zfill(2) != "01" or "PRESIDENTE" not in fila["CORNOMBRE"].upper():
                    raise AssertionError("Corporación histórica inesperada")
                if not fila["VOTOS"].isdigit():
                    raise ValueError(f"Votos no enteros en línea {nacionales + 1}")
                registros.append({"_id_fuente": fuente["id_fuente"],
                                  "_tipo_resultado": "por_verificar_en_metadatos",
                                  "_miembro_origen": miembro, **fila})
    tabla = pd.DataFrame(registros)
    if tabla.empty:
        raise AssertionError("El CSV histórico no contiene Barrancabermeja, Santander")
    codigos = tabla[["DEP", "MUN"]].drop_duplicates()
    if len(codigos) != 1 or tuple(codigos.iloc[0]) != (COD_DEPTO, COD_MUNICIPIO):
        raise AssertionError(f"Códigos de Barrancabermeja no corresponden a {COD_DEPTO}/{COD_MUNICIPIO}")
    llave = ["DEP", "MUN", "ZONA", "PUESTO", "MESA", "CORCODIGO", "CIR", "PAR", "CAN"]
    if tabla.duplicated(llave).any():
        raise AssertionError("Hay llaves mesa/opción duplicadas en Presidencia 2022")
    destino = (RAIZ / "datos" / "intermedios" / "presidencia" / "2022" /
               fuente["id_fuente"] / (fuente["id_fuente"].lower() + "_barrancabermeja.csv"))
    destino.parent.mkdir(parents=True, exist_ok=True)
    temporal = destino.with_suffix(".csv.part")
    try:
        tabla.to_csv(temporal, index=False, encoding="utf-8-sig")
        temporal.replace(destino)
    finally:
        temporal.unlink(missing_ok=True)
    return {"id_fuente": fuente["id_fuente"], "estado": "extraida_pendiente_perfilado",
            "filas_nacionales": nacionales, "filas_locales": len(tabla),
            "mesas_locales": len(tabla[["ZONA", "PUESTO", "MESA"]].drop_duplicates()),
            "votos_locales": int(tabla.VOTOS.astype("int64").sum()),
            "codigos": {"DEP": COD_DEPTO, "MUN": COD_MUNICIPIO},
            "tipo_resultado": "por_verificar_en_metadatos",
            "ruta_relativa": destino.relative_to(RAIZ).as_posix(),
            "sha256": sha256_archivo(destino), "miembro": miembro}

qa_2022 = {}
for f, d in zip(FUENTES, adquisiciones):
    if f["id_fuente"] not in {"REG_2022_PRES_1V", "REG_2022_PRES_2V"}:
        continue
    if d["estado"] not in {"existente_validado", "descargado_validado", "manual_validado"}:
        qa_2022[f["id_fuente"]] = {"estado": "sin_original"}
        continue
    try:
        qa_2022[f["id_fuente"]] = extraer_presidencia_2022(f, d)
    except (OSError, ValueError, AssertionError, UnicodeError, zipfile.BadZipFile) as exc:
        qa_2022[f["id_fuente"]] = {"estado": "pendiente_interpretacion",
                                 "detalle": f"{type(exc).__name__}: {exc}"}
(AUDITORIA / "control_presidencia_2022.json").write_text(
    json.dumps(qa_2022, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([{"id_fuente": k, "estado": v["estado"],
                       "filas_locales": v.get("filas_locales", 0),
                       "detalle": v.get("detalle", "")} for k, v in qa_2022.items()]))

,id_fuente,estado,filas_locales,detalle
0,REG_2022_PRES_1V,extraida_pendiente_perfilado,3473,
1,REG_2022_PRES_2V,extraida_pendiente_perfilado,1957,


In [14]:
def extraer_serie_municipal_dane(fuente: dict, adquisicion: dict) -> dict:
    """Lee las dos estructuras observadas; conserva la tabla ancha sin estimar población."""
    from openpyxl import load_workbook

    moderna = fuente["id_fuente"] == "DANE_MUN_EDAD_2018_2042"
    hoja_esperada, fila_enc, inicio, fin = (
        ("PobMunicipalxÁreaSexoEdad", 9, 2018, 2042) if moderna else
        ("NuevaMpal", 12, 2005, 2017))
    ruta = RAIZ / adquisicion["ruta_relativa"]
    libro = load_workbook(ruta, read_only=True, data_only=True)
    try:
        if hoja_esperada not in libro.sheetnames:
            raise ValueError("No existe la hoja esperada: " + hoja_esperada)
        hoja = libro[hoja_esperada]
        rows = hoja.iter_rows(values_only=True)
        encabezado_superior, encabezado = None, None
        registros = []
        for numero, row in enumerate(rows, start=1):
            if moderna and numero == 8:
                encabezado_superior = row
            if numero == fila_enc:
                encabezado = row
                break
        if encabezado is None or len(encabezado) < 20:
            raise ValueError("El encabezado DANE no tiene las columnas de edad esperadas")
        base = (encabezado_superior if moderna else encabezado)
        if [str(x).strip().upper() for x in base[:6]] != [
            "DP", "DPNOM", "MPIO" if moderna else "DPMP",
            "DPMP" if moderna else "MPIO", "AÑO", "ÁREA GEOGRÁFICA"]:
            raise ValueError("Los primeros seis campos no coinciden con el esquema DANE observado")
        nombres = [str(x).strip() for x in base[:6]]
        usados = {canon(x) for x in nombres}
        for i in range(6, max(len(encabezado), len(base))):
            etiqueta = (encabezado[i] if i < len(encabezado) else None)
            if etiqueta is None or not str(etiqueta).strip():
                etiqueta = base[i] if i < len(base) else None
            normal = canon(etiqueta) if etiqueta is not None else ""
            if not normal:
                normal = f"COLUMNA_{i + 1:03d}"
            nombre = normal
            if nombre in usados:
                nombre = f"{normal}_COL_{i + 1:03d}"
            usados.add(nombre)
            nombres.append(nombre)
        if len(nombres) < 100:
            raise ValueError("Menos de 100 columnas de población por sexo y edad")

        for numero, row in enumerate(rows, start=fila_enc + 1):
            primera = ["" if x is None else str(x).strip() for x in row[:6]]
            if len(primera) < 6 or primera[0] != "68":
                continue
            if canon(primera[1]) != "SANTANDER":
                raise AssertionError(f"DP=68 con departamento discordante, fila {numero}")
            par_municipio = primera[2:4]
            if "68081" not in par_municipio and "Barrancabermeja" not in par_municipio:
                continue
            if set(canon(v) for v in par_municipio) != {"68081", "BARRANCABERMEJA"}:
                raise AssertionError(f"Nombre y código DANE discordantes, fila {numero}")
            if not primera[4].isdigit() or not inicio <= int(primera[4]) <= fin:
                raise ValueError(f"Año fuera de la serie {inicio}-{fin}, fila {numero}")
            valores = ["" if x is None else str(x).strip() for x in row[:len(nombres)]]
            valores += [""] * (len(nombres) - len(valores))
            if any(not v.isdigit() for v in valores[6:] if v):
                raise ValueError(f"Población no entera en fila {numero}")
            registros.append({"_id_fuente": fuente["id_fuente"], "_hoja": hoja_esperada,
                              "_fila_excel": numero, **dict(zip(nombres, valores))})
    finally:
        libro.close()
    tabla = pd.DataFrame(registros)
    esperado = 3 * (fin - inicio + 1)
    if len(tabla) != esperado:
        raise AssertionError(f"Filas locales {len(tabla)}; esperadas {esperado} (3 áreas por año)")
    areas = {"CABECERA_MUNICIPAL", "CENTROS_POBLADOS_Y_RURAL_DISPERSO", "TOTAL"}
    tabla["_area_control"] = tabla["ÁREA GEOGRÁFICA"].map(canon)
    for anio, grupo in tabla.groupby("AÑO"):
        if len(grupo) != 3 or set(grupo["_area_control"]) != areas:
            raise AssertionError(f"Cobertura por área incompleta en {anio}")
    if set(tabla["AÑO"].astype(int)) != set(range(inicio, fin + 1)):
        raise AssertionError("La serie municipal no cubre todos los años declarados")
    columnas_valores = [c for c in nombres[6:] if tabla[c].ne("").any()]
    discrepancias = []
    for anio, grupo in tabla.groupby("AÑO"):
        por_area = grupo.set_index("_area_control")
        for col in columnas_valores:
            vals = por_area[col].to_dict()
            if all(vals.get(a, "").isdigit() for a in areas):
                diferencia = (int(vals["CABECERA_MUNICIPAL"]) +
                              int(vals["CENTROS_POBLADOS_Y_RURAL_DISPERSO"]) -
                              int(vals["TOTAL"]))
                if abs(diferencia) > 1:
                    discrepancias.append((anio, col, diferencia))
    if discrepancias:
        raise AssertionError(f"Totales por área discordantes: {discrepancias[:5]}")
    tabla = tabla.drop(columns=["_area_control"])
    destino = (RAIZ / "datos" / "intermedios" / "poblacion" / "serie" /
               fuente["id_fuente"] / (fuente["id_fuente"].lower() + "_barrancabermeja.csv"))
    destino.parent.mkdir(parents=True, exist_ok=True)
    temporal = destino.with_suffix(".csv.part")
    try:
        tabla.to_csv(temporal, index=False, encoding="utf-8-sig")
        temporal.replace(destino)
    finally:
        temporal.unlink(missing_ok=True)
    return {"id_fuente": fuente["id_fuente"], "estado": "extraida_pendiente_perfilado",
            "filas_locales": len(tabla), "anios": [inicio, fin], "areas": sorted(areas),
            "columnas": len(nombres), "columnas_numericas_con_datos": len(columnas_valores),
            "max_discrepancia_area_admitida": 1, "discrepancias_mayores_a_uno": 0,
            "ruta_relativa": destino.relative_to(RAIZ).as_posix(),
            "sha256": sha256_archivo(destino)}

qa_dane = {}
for f, d in zip(FUENTES, adquisiciones):
    if f["id_fuente"] not in {"DANE_MUN_EDAD_2018_2042", "DANE_MUN_EDAD_2005_2017"}:
        continue
    if d["estado"] not in {"existente_validado", "descargado_validado", "manual_validado"}:
        qa_dane[f["id_fuente"]] = {"estado": "sin_original"}
        continue
    try:
        qa_dane[f["id_fuente"]] = extraer_serie_municipal_dane(f, d)
    except (OSError, ValueError, AssertionError, ImportError, KeyError) as exc:
        qa_dane[f["id_fuente"]] = {"estado": "pendiente_interpretacion",
                                  "detalle": f"{type(exc).__name__}: {exc}"}
(AUDITORIA / "control_poblacion_dane.json").write_text(
    json.dumps(qa_dane, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([{"id_fuente": k, "estado": v["estado"],
                       "filas_locales": v.get("filas_locales", 0),
                       "detalle": v.get("detalle", "")} for k, v in qa_dane.items()]))

,id_fuente,estado,filas_locales,detalle
0,DANE_MUN_EDAD_2018_2042,extraida_pendiente_perfilado,75,
1,DANE_MUN_EDAD_2005_2017,extraida_pendiente_perfilado,39,


In [15]:
# Un único inventario de salidas locales, incluidos los adaptadores específicos.
inventario_adaptadores = []
for id_fuente, qa_fuente in {**qa_2022, **qa_dane}.items():
    if qa_fuente.get("estado") == "extraida_pendiente_perfilado":
        inventario_adaptadores.append({"id_fuente": id_fuente, "estado": qa_fuente["estado"],
            "ruta_relativa": qa_fuente["ruta_relativa"], "filas_locales": qa_fuente["filas_locales"],
            "sha256": qa_fuente["sha256"]})
for id_fuente, qa_fuente, estado in [
    ("REG_2026_PRES_1V", qa_piloto_generado, "conciliada"),
    ("REG_2026_PRES_2V", qa_2v, "conciliada"),
    ("REG_2026_CONGRESO", qa_congreso_2026, "extraida_pendiente_perfilado")]:
    if not qa_fuente.get("salidas"):
        continue
    for nombre, ruta_relativa in qa_fuente["salidas"].items():
        inventario_adaptadores.append({"id_fuente": id_fuente, "estado": estado,
            "tabla": nombre, "ruta_relativa": ruta_relativa,
            "sha256": sha256_archivo(RAIZ / ruta_relativa)})
tabla_locales = pd.concat([tabla_locales, pd.DataFrame(inventario_adaptadores)], ignore_index=True)
print("Tablas locales registradas en inventario:", len(tabla_locales))

Tablas locales registradas en inventario: 17


### 7.2 Diagnóstico de formatos pendientes

Para revisar una ejecución, consulta el cuaderno y los archivos de estado y diagnóstico de la misma carpeta de auditoría. Comprueba que tengan el mismo ID. No combines archivos de ejecuciones distintas.

El archivo `diagnostico_estructuras_para_adaptadores.json` registra nombres y tamaños de archivos, muestras de registros, encabezados, filas municipales candidatas en XLSX y enlaces del selector. Para inspeccionar el ZIP de Congreso 2026 se requiere espacio temporal adicional; el archivo temporal se elimina al finalizar. El diagnóstico aporta muestras para interpretar las fuentes que sigan pendientes.

In [16]:
# Diagnóstico compacto de recursos aún sin adaptador. No modifica los originales.
def muestra_tabular(stream, max_lineas: int = 4) -> dict:
    lineas = []
    for _ in range(max_lineas):
        linea = stream.readline(4096)
        if not linea:
            break
        lineas.append(linea)
    primer = lineas[0] if lineas else b""
    codec = ("utf-16" if primer.startswith((b"\xff\xfe", b"\xfe\xff")) else
             "utf-16-le" if primer.count(b"\x00") > len(primer) // 4 else "utf-8-sig")
    try:
        texto = [x.decode(codec) for x in lineas]
    except UnicodeDecodeError:
        codec, texto = "cp1252", [x.decode("cp1252") for x in lineas]
    return {"codificacion_muestra": codec, "longitudes_bytes": [len(x.rstrip(b"\r\n")) for x in lineas],
            "lineas_iniciales": [x.strip()[:400] for x in texto],
            "separadores_primera": {s: texto[0].count(s) for s in (";", ",", "\t", "|")}
            if texto else {}}

def diagnosticar_archivo(f: dict, d: dict) -> dict:
    ruta = RAIZ / d["ruta_relativa"]
    base = {"id_fuente": f["id_fuente"], "sha256": d["sha256"],
            "ruta_relativa": d["ruta_relativa"], "extension": f["extension"]}
    if f["extension"] == "csv":
        with ruta.open("rb") as stream:
            base["muestra"] = muestra_tabular(stream)
        return base
    if f["extension"] == "xlsx":
        from openpyxl import load_workbook
        libro = load_workbook(ruta, read_only=True, data_only=True)
        try:
            hojas = []
            for hoja in libro:
                comienzo, ciudad = [], []
                for numero, row in enumerate(hoja.iter_rows(values_only=True), start=1):
                    vals = ["" if v is None else str(v)[:100] for v in row]
                    if len(comienzo) < 16 and any(vals):
                        comienzo.append({"fila_excel": numero, "valores": vals[:65]})
                    if len(ciudad) < 3 and any(
                        canon(v) in {"BARRANCABERMEJA", "68081", "68081_0"} for v in vals
                    ):
                        ciudad.append({"fila_excel": numero, "valores": vals[:80]})
                    if numero >= 100_000 or (len(comienzo) >= 16 and len(ciudad) >= 3):
                        break
                hojas.append({"nombre": hoja.title, "filas_max_declaradas": hoja.max_row,
                              "columnas_max_declaradas": hoja.max_column,
                              "filas_iniciales": comienzo, "candidatas_municipio": ciudad})
            base["hojas"] = hojas
        finally:
            libro.close()
        return base
    with zipfile.ZipFile(ruta) as z:
        miembros = []
        for m in z.infolist():
            if m.is_dir():
                continue
            entry = {"nombre": m.filename, "bytes_descomprimidos": m.file_size,
                     "crc32": f"{m.CRC:08x}"}
            nombre = PurePosixPath(m.filename).name
            if nombre.lower().endswith((".csv", ".txt", ".tsv")) and not re.search(r"(?i)^hash", nombre):
                with z.open(m) as stream:
                    entry["muestra"] = muestra_tabular(stream)
            elif nombre.lower().endswith(".zip"):
                entry["estructura_anidada"] = "ZIP anidado: inventario y primeras líneas"
                temporal_anidado = None
                try:
                    if m.file_size <= 16 * 1024**2:
                        contenido = io.BytesIO(z.read(m))
                    else:
                        if m.file_size > 650 * 1024**2 or shutil.disk_usage(AUDITORIA).free < m.file_size + 512 * 1024**2:
                            raise OSError("Sin espacio reservado para inspección del ZIP anidado")
                        with tempfile.NamedTemporaryFile(dir=AUDITORIA, suffix=".zip.part", delete=False) as tmp:
                            temporal_anidado = Path(tmp.name)
                            with z.open(m) as stream:
                                shutil.copyfileobj(stream, tmp, length=1024 * 1024)
                        contenido = temporal_anidado
                    with zipfile.ZipFile(contenido) as interno:
                        detalles = []
                        for item in interno.infolist()[:60]:
                            if item.is_dir():
                                continue
                            detalle = {"nombre": item.filename,
                                       "bytes_descomprimidos": item.file_size,
                                       "crc32": f"{item.CRC:08x}"}
                            subnombre = PurePosixPath(item.filename).name.lower()
                            if subnombre.endswith((".csv", ".txt", ".tsv")):
                                with interno.open(item) as stream:
                                    detalle["muestra"] = muestra_tabular(stream, max_lineas=3)
                            detalles.append(detalle)
                        entry["miembros_anidados"] = detalles
                except (OSError, ValueError, zipfile.BadZipFile) as exc:
                    entry["error_anidado"] = f"{type(exc).__name__}: {exc}"
                finally:
                    if temporal_anidado is not None:
                        temporal_anidado.unlink(missing_ok=True)
            elif nombre.lower().endswith(".xlsx") and m.file_size <= 120 * 1024**2:
                try:
                    from openpyxl import load_workbook
                    workbook = load_workbook(io.BytesIO(z.read(m)), read_only=True, data_only=True)
                    try:
                        entry["hojas_anidadas"] = []
                        for sh in workbook:
                            primeros = []
                            for numero, row in enumerate(sh.iter_rows(values_only=True), start=1):
                                if any(v is not None for v in row):
                                    primeros.append({"fila": numero,
                                                    "valores": ["" if v is None else str(v)[:80] for v in row[:24]]})
                                if numero >= 25 or len(primeros) >= 8:
                                    break
                            entry["hojas_anidadas"].append({"nombre": sh.title,
                                "filas_max_declaradas": sh.max_row, "columnas_max_declaradas": sh.max_column,
                                "filas_iniciales": primeros})
                    finally:
                        workbook.close()
                except (OSError, ValueError, zipfile.BadZipFile) as exc:
                    entry["error_xlsx_anidado"] = f"{type(exc).__name__}: {exc}"
            miembros.append(entry)
        base["miembros"] = miembros
    return base

diagnosticos_pendientes = []
for fuente, adquisicion in zip(FUENTES, adquisiciones):
    if fuente["id_fuente"] == "REG_2026_PRES_1V" or (
        fuente["id_fuente"] == "REG_2026_PRES_2V" and qa_2v.get("estado") == "conciliada"
    ) or (fuente["id_fuente"] == "REG_2026_CONGRESO" and
          qa_congreso_2026.get("estado") == "extraida_pendiente_perfilado"
    ) or qa_2022.get(fuente["id_fuente"], {}).get("estado") == "extraida_pendiente_perfilado" or (
        qa_dane.get(fuente["id_fuente"], {}).get("estado") == "extraida_pendiente_perfilado"
    ):
        continue
    if adquisicion["estado"] not in {"existente_validado", "descargado_validado", "manual_validado"}:
        continue
    try:
        diagnosticos_pendientes.append(diagnosticar_archivo(fuente, adquisicion))
    except (OSError, ValueError, UnicodeError, zipfile.BadZipFile) as exc:
        diagnosticos_pendientes.append({"id_fuente": fuente["id_fuente"],
                                       "error": f"{type(exc).__name__}: {exc}"})

paquete = {"ejecucion": ID_EJECUCION, "fuentes": diagnosticos_pendientes,
           "resolucion_selector": descubrimientos,
           "enlaces_portal": enlaces_visibles}
ruta_diagnostico = AUDITORIA / "diagnostico_estructuras_para_adaptadores.json"
ruta_diagnostico.write_text(json.dumps(paquete, ensure_ascii=False, indent=2), encoding="utf-8")
print("Diagnóstico de estructuras:", ruta_diagnostico.relative_to(RAIZ))
print("Fuentes inspeccionadas:", len(diagnosticos_pendientes))
display(pd.DataFrame([{"id_fuente": x["id_fuente"],
                       "miembros_u_hojas": len(x.get("miembros", x.get("hojas", []))),
                       "error": x.get("error", "")} for x in diagnosticos_pendientes]))

Diagnóstico de estructuras: docs\fuentes\auditorias\00_fuentes_y_adquisicion\20261003_164243_810157\diagnostico_estructuras_para_adaptadores.json
Fuentes inspeccionadas: 2


,id_fuente,miembros_u_hojas,error
0,REG_2018_PRES_1V,1,
1,REG_2018_PRES_2V,1,


## 8. Estado del corpus

Por fuente, el informe registra la página o URL, la validación del original, las tablas encontradas, las filas extraídas, los pendientes y la cobertura. `REG_2026_PRES_1V` se valida al conciliar las 566 mesas. El censo histórico requiere datos de 2019, 2022, 2023 y 2026. Las fuentes nuevas quedan pendientes de perfilado. El estado diferencia originales completos, tablas locales y fuentes listas para análisis.

La etapa sigue abierta mientras falten fuentes centrales o haya formatos sin interpretar. Para contar candidaturas con y sin votos se necesitan los registros de inscripción de Alcaldía y Concejo de 2019 y 2023. Cada exportación debe incluir ambas corporaciones del año. Las fuentes complementarias, como las de 2015, el CNPV detallado y la cartografía, permanecen en el inventario con su estado.

In [17]:
OK = {"existente_validado", "descargado_validado", "manual_validado"}
estado_final = []
for f, d in zip(FUENTES, adquisiciones):
    source_rows = [r for r in diagnosticos_genericos if r.get("id_fuente") == f["id_fuente"]]
    tab = [r for r in tablas_genericas if r["id_fuente"] == f["id_fuente"]]
    if f["id_fuente"] == "REG_2026_PRES_1V":
        estado_tabla = "conciliada" if qa_piloto_generado.get("criterio_avance_piloto") == "aprobado" else "pendiente"
        filas = qa_piloto_generado.get("escrutinio", {}).get("filas_locales", 0)
    elif f["id_fuente"] == "REG_2026_PRES_2V":
        estado_tabla = ("sin_original" if d["estado"] not in OK else
                        "conciliada" if qa_2v.get("estado") == "conciliada" else
                        "pendiente_interpretacion")
        filas = qa_2v.get("escrutinio", {}).get("filas_locales", 0)
    elif f["id_fuente"] == "REG_2026_CONGRESO":
        estado_tabla = ("sin_original" if d["estado"] not in OK else
                        "extraida_pendiente_perfilado" if qa_congreso_2026.get("estado") == "extraida_pendiente_perfilado" else
                        "pendiente_interpretacion")
        filas = qa_congreso_2026.get("filas_escrutinio_locales", 0)
    elif f["id_fuente"] in qa_2022:
        q = qa_2022[f["id_fuente"]]
        estado_tabla = q["estado"] if d["estado"] in OK else "sin_original"
        filas = q.get("filas_locales", 0)
    elif f["id_fuente"] in qa_dane:
        q = qa_dane[f["id_fuente"]]
        estado_tabla = q["estado"] if d["estado"] in OK else "sin_original"
        filas = q.get("filas_locales", 0)
    elif not d["estado"] in OK:
        estado_tabla, filas = "sin_original", 0
    elif not tab:
        estado_tabla, filas = "pendiente_interpretacion", 0
    elif any(x.get("estado") not in {"extraido_pendiente_perfilado"} for x in source_rows):
        estado_tabla, filas = "extraccion_parcial", sum(x.get("filas_locales", 0) for x in tab)
    else:
        estado_tabla, filas = "extraida_pendiente_perfilado", sum(x.get("filas_locales", 0) for x in tab)
    if f["id_fuente"] == "REG_2026_CONGRESO" and qa_congreso_2026.get("estado") == "extraida_pendiente_perfilado":
        cobertura, evidencia_cobertura = "corporaciones_identificadas", "CAM, SEN; escrutinio con código de corporación"
    else:
        cobertura, evidencia_cobertura = seleccionar_cobertura(f, tab, source_rows)
    if estado_tabla == "extraida_pendiente_perfilado" and cobertura in {
        "corporaciones_no_verificadas", "anios_no_verificados"
    }:
        estado_tabla = "cobertura_no_verificada"
    if d["estado"] not in OK:
        accion = ("El enlace visible devuelve HTML: exportar Santander desde el selector oficial; "
                  "registrar archivo y evidencia manual"
                  if f["id_fuente"] == "REG_2022_CONGRESO" and "página HTML" in d["detalle"] else
                  "Resolver la URL o descargar desde la página oficial y registrar ARCHIVOS_MANUALES"
                  if not f["url_directa"] else "Revisar el detalle de adquisición y volver a intentar")
    elif d["estado_procedencia"] == "pendiente":
        accion = "Registrar EVIDENCIA_MANUAL: URL oficial, o página y fecha UTC de la descarga"
    elif estado_tabla == "pendiente_interpretacion":
        accion = (qa_2v.get("detalle", "") if f["id_fuente"] == "REG_2026_PRES_2V" else
                  qa_congreso_2026.get("detalle", "") if f["id_fuente"] == "REG_2026_CONGRESO" else
                  qa_2022.get(f["id_fuente"], qa_dane.get(f["id_fuente"], {})).get("detalle", "") or
                  "Inspeccionar diagnostico_estructuras_para_adaptadores.json y agregar adaptador verificado")
    elif estado_tabla == "cobertura_no_verificada":
        accion = evidencia_cobertura + "; revisar fuente y datos locales"
    elif estado_tabla == "extraccion_parcial":
        accion = "Resolver miembros u hojas sin interpretación antes del perfilado"
    else:
        accion = "Perfilado en 01 tras cerrar todas las fuentes centrales"
    estado_final.append({"id_ejecucion": ID_EJECUCION,
                         "id_fuente": f["id_fuente"], "familia": f["familia"],
                         "anio": f["anio"], "obligatoria": f["obligatoria"],
                         "estado_original": d["estado"], "estado_tabla_local": estado_tabla,
                         "estado_procedencia": d["estado_procedencia"],
                         "estado_cobertura": cobertura, "evidencia_cobertura": evidencia_cobertura,
                         "filas_locales": filas, "url_origen": d["url_origen"],
                         "pagina_oficial": f["pagina_oficial"], "detalle": d["detalle"],
                         "accion_siguiente": accion})
estado_corpus = pd.DataFrame(estado_final)
centrales = estado_corpus[estado_corpus.obligatoria]
originales_centrales = bool(centrales.estado_original.isin(OK).all() and
                           centrales.estado_procedencia.ne("pendiente").all())
tablas_centrales_presentes = bool(centrales.estado_tabla_local.isin({
    "conciliada", "extraida_pendiente_perfilado"}).all())
fuentes_ampliadas_resueltas = bool(estado_corpus.loc[~estado_corpus.obligatoria,
    "estado_original"].isin(OK).all())
listo_para_01 = originales_centrales and tablas_centrales_presentes

catalogo.to_csv(AUDITORIA / "catalogo_fuentes.csv", index=False, encoding="utf-8-sig")
tabla_adquisiciones.to_csv(AUDITORIA / "adquisiciones.csv", index=False, encoding="utf-8-sig")
tabla_miembros.to_csv(AUDITORIA / "inventario_miembros.csv", index=False, encoding="utf-8-sig")
tabla_esquemas.to_json(AUDITORIA / "diagnostico_esquemas.json", orient="records",
                      indent=2, force_ascii=False)
tabla_locales.to_csv(AUDITORIA / "tablas_locales.csv", index=False, encoding="utf-8-sig")
estado_corpus.to_csv(AUDITORIA / "estado_corpus.csv", index=False, encoding="utf-8-sig")
bitacora = {"id_ejecucion": ID_EJECUCION, "fecha_utc": FECHA_INICIO.isoformat(),
            "codigo_dane_barrancabermeja": "68081",
            "codigo_electoral_2026_1v": qa_piloto_generado.get("codigo_electoral"),
            "control_2026_2v": {k: qa_2v.get(k) for k in ("estado", "detalle", "mesas_divipol")},
            "control_congreso_2026": {k: qa_congreso_2026.get(k) for k in
                                      ("estado", "detalle", "filas_preconteo_locales", "filas_escrutinio_locales")},
            "control_presidencia_2022": {k: v.get("estado") for k, v in qa_2022.items()},
            "control_poblacion_dane": {k: v.get("estado") for k, v in qa_dane.items()},
            "originales_centrales_completos": originales_centrales,
            "tablas_centrales_presentes": tablas_centrales_presentes,
            "fuentes_ampliadas_resueltas": fuentes_ampliadas_resueltas,
            "listo_para_cuaderno_01": listo_para_01,
            "pendientes_centrales": centrales.loc[
                ~centrales.estado_tabla_local.isin({"conciliada", "extraida_pendiente_perfilado"}) |
                centrales.estado_procedencia.eq("pendiente"),
                "id_fuente"].tolist(),
            "pendientes_ampliacion": estado_corpus.loc[
                ~estado_corpus.obligatoria & ~estado_corpus.estado_original.isin(OK),
                "id_fuente"].tolist(),
            "advertencia": "CSV nuevos filtrados nominalmente requieren perfilado y QA; no se declara escrutinio definitivo por nombre de archivo"}
(AUDITORIA / "estado_corpus.json").write_text(
    json.dumps(bitacora, ensure_ascii=False, indent=2), encoding="utf-8")
display(estado_corpus[["id_fuente", "obligatoria", "estado_original", "estado_tabla_local", "filas_locales"]])
print("Originales centrales completos:", originales_centrales)
print("Tablas locales centrales presentes:", tablas_centrales_presentes)
print("Listo para el cuaderno 01:", listo_para_01)
print("Pendientes centrales:", ", ".join(bitacora["pendientes_centrales"]) or "ninguno")
print("Auditoría:", AUDITORIA.relative_to(RAIZ))
print("ID de ejecución para cotejar JSON y CSV:", ID_EJECUCION)

,id_fuente,obligatoria,estado_original,estado_tabla_local,filas_locales
0,REG_2026_PRES_1V,True,existente_validado,conciliada,9056
1,REG_2026_PRES_2V,True,existente_validado,conciliada,2830
2,REG_2026_CONGRESO,True,existente_validado,extraida_pendiente_perfilado,833000
3,REG_2023_TERR,True,sin_url_resuelta,sin_original,0
4,REG_2019_TERR,True,sin_url_resuelta,sin_original,0
5,REG_2022_PRES_1V,True,existente_validado,extraida_pendiente_perfilado,3473
6,REG_2022_PRES_2V,True,existente_validado,extraida_pendiente_perfilado,1957
7,REG_2022_CONGRESO,True,error_adquisicion,sin_original,0
8,REG_CENSO_HIST,True,sin_url_resuelta,sin_original,0
9,DANE_MUN_EDAD_2018_2042,True,existente_validado,extraida_pendiente_perfilado,75


Originales centrales completos: False
Tablas locales centrales presentes: False
Listo para el cuaderno 01: False
Pendientes centrales: REG_2023_TERR, REG_2019_TERR, REG_2022_CONGRESO, REG_CENSO_HIST, REG_CANDIDATURAS_2019, REG_CANDIDATURAS_2023
Auditoría: docs\fuentes\auditorias\00_fuentes_y_adquisicion\20261003_164243_810157
ID de ejecución para cotejar JSON y CSV: 20261003_164243_810157
